# Prueba del ADRF6850 por SPI

Sigue una secuencia de pasos y, **antes de cada operación, imprime el resultado esperado** para contrastarlo con lo que realmente pasa. Sirve para dos pruebas con la misma secuencia:

| `MODO` | Quién hace de ADRF6850 | Con qué se contrasta |
|---|---|---|
| `'ESP32'` | el emulador (`adrf6850_emulator.ino`) | las líneas del Serial del ESP32 y el LED azul (= D3) |
| `'ADRF_REAL'` | el ADRF6850 de la placa | **mediciones de banco** (osciloscopio, tester, analizador de espectro) y el LED D3 |

## Qué hace `INTERACTIVO`

- `INTERACTIVO = False` (por defecto): no se detiene nunca (sirve para `Run All`). Las verificaciones manuales y las mediciones solo se **listan**, como plan de prueba.
- `INTERACTIVO = True`: el notebook **pausa** en cada punto.
  - Verificaciones manuales (LED, mirar el Serial…): pregunta `s` / `n` / Enter.
  - **Mediciones de banco** (solo `ADRF_REAL`): muestra qué instrumento usar, cómo conectarlo y el valor esperado con su tolerancia, y abre un **cuadro de texto para escribir el valor medido** (en VS Code aparece arriba de la ventana; en Jupyter, debajo de la celda). Acepta `3,31` o `3.31`; Enter omite la medición y `n` indica que no pudiste medir. Con el valor ingresado calcula **OK / FUERA** y, si queda fuera, pide un comentario opcional.
- **Registro**: toda la prueba (verificaciones automáticas, manuales y mediciones, con su hora, criterio y resultado) se guarda en un **CSV con timestamp** en la carpeta `resultados_adrf/` (para Excel en español: separador `;` y coma decimal). Se va escribiendo fila por fila durante la corrida, así que si algo falla a mitad de camino no se pierde lo medido. La celda de resumen imprime la tabla y la ruta del archivo.

Conviene correr una vez con `INTERACTIVO = False` para leer el plan completo, y recién después en el laboratorio con `INTERACTIVO = True`.

## Configuración y enlace SPI

La configuración inicial del ADRF no tiene valores por defecto en el driver. `CONFIG_ORIGEN` elige de dónde sale (todo lo esperado se calcula de ella):

- `'servidor'`: el notebook trae con `adrf_get_conf` el `adrf6850.conf` de la Red Pitaya, y el init usa ese mismo archivo (`adrf_init`).
- `'cliente'`: usa el diccionario `CONF_CLIENTE` y lo manda con `adrf_init_cfg`; el servidor puede arrancar sin `-c`.

El CSV registra el origen y, con `'servidor'`, el `crc32` del archivo.

Cada registro que escribe el driver se lee de vuelta por `SDO` y, si no coincide, se reescribe hasta `verify_retries` veces (0 a 3, en el `.conf` o en `CONF_CLIENTE`); si no coincide ni así, el init falla y dice qué registro fue. `adrf_get_verify` trae las escrituras verificadas, los reintentos y la última falla de la última operación: con el enlace sano no hay reintentos. Hace falta `SDO` cableado (`J3` pines 9-10, al pin 4 del `E2` del Red Pitaya), salvo con `verify_retries = 0`.

En `ADRF_REAL`, `VGAIN` lo aplica el DAC de la Red Pitaya (`OUT2`, canal B del buffer de TX) a través del servidor de datos (`PORT_DATOS`): el notebook calcula la tensión con el driver y la aplica solo, sin pedirla. Antes de conectar `OUT2` a `J2`, la sección 0b verifica la tensión del DAC con el osciloscopio y lo calibra (`VERIFICAR_DAC`; una vez calibrado se pone en `False`). Con `VGAIN_POR_DAC = False` la tensión se aplica a mano con una fuente externa.

## Qué prueba este notebook y qué no

Las verificaciones automáticas (`[OK]`/`[FALLO]`) comparan lo que devuelve el servidor con valores calculados acá de forma independiente, a partir de las ecuaciones de la hoja de datos. Pero `adrf_get_state` es la copia interna del driver, no el chip, y el `ack` solo dice que el driver terminó sin error. La evidencia de que el chip recibió y usa la configuración es externa:

- **Lectura de registros por `SDO`** (secciones 2a y 2d): lo escrito por `SDI` vuelve por `SDO`.
- **ESP32**: el Serial muestra cada transacción, el LO decodificado y si el init es conforme a la hoja de datos.
- **ADRF real**: `LDET` con el tester (sección 2c; el LED D3 puede no encenderse aunque el PLL enganche, si su tensión directa supera lo que entrega `LDET`), la corriente de alimentación, `VTUNE`, la frecuencia y el nivel en `LOMON` y el tono de banda base. El detector de lock se habilita por SPI (`CR23` bit 4).

## Leyenda de lo que se imprime

- `[OK]` / `[FALLO]`: verificación automática (valor esperado calculado acá vs. valor real del servidor).
- `RP    >` líneas esperadas en la consola del servidor del Red Pitaya, en ese orden.
- `ESP32 >` líneas esperadas en el Serial del ESP32, en ese orden (`ESP32 ~` = sin orden estricto).
- `[MANUAL]` algo que hay que mirar (LED, Serial).
- `[MEDIR]` una medición de banco, con instrumento, forma de medir y criterio.

## Secuencia de laboratorio

**Con el ESP32**: (1) cargar el sketch; (2) con todo apagado, cablear E2→ESP32 (pin 3→GPIO23, pin 4←GPIO19, pin 5→GPIO18, pin 6→GPIO5, GND común); (3) Ethernet Red Pitaya↔PC; (4) energizar el Red Pitaya; (5) verificar `ls /dev/spidev*`, compilar `gcc -O2 -Wall adrf_server.c adrf6850.c adrf6850_conf.c -o adrf_server` y arrancar `./adrf_server -c adrf6850.conf -i 300` (o sin `-c` si `CONFIG_ORIGEN = 'cliente'`; el `-i 300` agrega una pausa de 300 µs tras cada transacción SPI, solo para el emulador ESP32; este servidor no usa la FPGA, así que no hace falta cargar el bitstream); (6) USB del ESP32 a la PC; (7) abrir el Serial Monitor (115200) y, si hace falta, pulsar EN para ver el banner; (8) correr este notebook. **No cierres ni reabras el monitor durante la prueba**: reiniciaría el ESP32 (equivale a cortar la alimentación del chip) y volvería a modo I2C.

**Con el ADRF real**: primero la sección 0 (alimentación, referencia, CS, `R10`), **sin** SPI conectado. Recién después conectar SPI y GND al Red Pitaya (solo cables individuales para CLK, SDI, SDO, CS y GND; **J3 pines 1-2 sin conectar**), arrancar `./adrf_server -c adrf6850.conf` (o sin `-c` si `CONFIG_ORIGEN = 'cliente'`) **sin** `-i` y sin `-a`: el notebook hace el init. Arrancar también `./adc_server`, el servidor de datos: ahí está el DAC que aplica `VGAIN`. `OUT2` se conecta a `J2` recién en la sección 0b, después de verificar la tensión del DAC con el osciloscopio. Correr con `MODO = 'ADRF_REAL'` e `INTERACTIVO = True`. Si repetís la corrida sin cortar la alimentación del ADRF, el cambio de modo I2C→SPI ya no se vuelve a probar (queda enganchado hasta cortar la alimentación).


In [29]:
# ============================================================
# Configuracion
# ============================================================
import csv
import os
import socket
import time
import numpy as np
import redpitaya_io as rpio
assert all(hasattr(rpio, f) for f in ('adrf_read_reg', 'adrf_get_conf', 'adrf_init_cfg', 'adrf_get_verify')), (
    "A redpitaya_io le faltan funciones del ADRF: usar la redpitaya_io.py de esta carpeta (si ya se importo otra, reiniciar el kernel)")

HOST = '192.168.1.100'    # IP del Red Pitaya
PORT = 1005               # puerto del servidor del ADRF (adrf_server); el servidor de datos usa el 1003

MODO = 'ADRF_REAL'            # 'ESP32'  o  'ADRF_REAL'
INTERACTIVO = True       # True: pausa en cada verificacion manual y en cada medicion de banco

# --- VGAIN ---
VGAIN_POR_DAC = (MODO == 'ADRF_REAL') # True: VGAIN por el DAC (OUT2 -> J2, servidor arrancado con -d) | False: a mano con una fuente externa
VERIFICAR_DAC = True                  # True: la seccion 0b calibra el DAC con el osciloscopio (OUT2 desconectado de J2) | False: ya calibrado (servidor arrancado con -F y -O) y OUT2 conectado

# --- Registro de resultados: un CSV con timestamp que se va escribiendo durante la prueba ---
DIR_RESULTADOS = 'resultados_adrf'
T_INICIO       = time.strftime("%Y%m%d_%H%M%S")
ARCHIVO_CSV    = os.path.join(DIR_RESULTADOS, f"prueba_{MODO.lower()}_{'interactivo' if INTERACTIVO else 'auto'}_{T_INICIO}.csv")

# --- Configuracion del ADRF: el driver no tiene valores por defecto, todo sale de aca ---
CONFIG_ORIGEN = 'servidor'   # 'servidor': el .conf de la Red Pitaya (adrf_init) | 'cliente': CONF_CLIENTE (adrf_init_cfg)

# Solo con CONFIG_ORIGEN = 'cliente'. Mismas claves que adrf6850.conf (ahi estan explicadas).
CONF_CLIENTE = dict(
    f_rf_hz=161_975_000, f_fi_hz=10_700_000, corr_ppm=0.0,          # LO = f_rf_hz - f_fi_hz, corregido por corr_ppm (o definir lo_hz)
    bb_wideband=0, bb_fc_mhz=30, vocm_interno=1,                    # vocm_interno = 1 es obligatorio en esta placa (VOCM a masa)
    cp_code=7, muxout=0, autocal_us=100, lock_timeout_us=2000,
    refin_hz=13_500_000, ref_doblador=1, ref_div2=0, ref_rdiv_en=0, ref_rdiv=1,   # refin_hz lo fija el oscilador U2
    lomon_en=0, lomon_pwr_dbm=-24, gain_mdb=40_000, vga_pendiente=0,
    verify_retries=3,                                               # 0 = no verifica (no hace falta SDO); 1 a 3 = reintentos
)

if CONFIG_ORIGEN == 'servidor':
    socket.setdefaulttimeout(5.0)
    try:
        _s = rpio.conectar_comandos(HOST, PORT)
        try:
            CONF = rpio.adrf_get_conf(_s)
        finally:
            _s.close()
    except (OSError, ValueError, RuntimeError) as e:
        raise RuntimeError(f"No se pudo traer el .conf del servidor ({HOST}:{PORT}): {e}. Hay que arrancar adrf_server con -c "
                           f"archivo.conf antes de abrir el notebook, o usar CONFIG_ORIGEN = 'cliente'.") from e
    finally:
        socket.setdefaulttimeout(None)
    CONF_ID = f"servidor, crc32 {CONF['_crc32']:08x}"
elif CONFIG_ORIGEN == 'cliente':
    CONF = dict(CONF_CLIENTE)
    CONF_ID = "cliente"
else:
    raise ValueError("CONFIG_ORIGEN debe ser 'servidor' o 'cliente'")

_c = lambda k: int(CONF[k])
REFIN_HZ, DOBLADOR, DIV2 = _c('refin_hz'), _c('ref_doblador'), _c('ref_div2')
RDIV_EN, RDIV = _c('ref_rdiv_en'), _c('ref_rdiv')
PFD_HZ        = REFIN_HZ * (1 + DOBLADOR) // ((RDIV if RDIV_EN else 1) * (1 + DIV2))
AUTOCAL_US    = _c('autocal_us')
BSCDIV        = (PFD_HZ * AUTOCAL_US) // 24_000_000
VOCM_INT, VGA_POL, MUXOUT = _c('vocm_interno'), _c('vga_pendiente'), _c('muxout')
BB_WIDE_DEF   = bool(_c('bb_wideband'))
BB_FC_DEF     = {50: 0, 43: 1, 37: 2, 30: 3}[_c('bb_fc_mhz')]
CP_DEFECTO    = _c('cp_code')                                     # "DEFECTO" = lo que dice la configuracion
GAIN_DEFECTO_MDB = _c('gain_mdb')
VERIFY_RETRIES = _c('verify_retries')
LOMON_EN_DEF  = _c('lomon_en')
LOMON_PWR_DEF = {-24: 0, -18: 1, -12: 2, -6: 3}[_c('lomon_pwr_dbm')]
F_RF_CONF     = CONF.get('f_rf_hz') or None
F_FI_CONF     = CONF.get('f_fi_hz') or None
CORR_PPM_CONF = float(CONF.get('corr_ppm', 0.0))
LO_DEFECTO_HZ = rpio.adrf_lo_desde_cfg(CONF)                      # igual que el servidor: redondeo al mas cercano
if F_RF_CONF and F_FI_CONF and not CONF.get('_lo_explicito'):     # el LO que calculo el servidor tiene que ser el que sale de f_rf, f_fi y corr_ppm
    assert LO_DEFECTO_HZ == int((F_RF_CONF - F_FI_CONF) / (1 + CORR_PPM_CONF * 1e-6) + 0.5), "el LO del servidor no coincide con f_rf_hz - f_fi_hz corregido"
CR29_CONF     = (VOCM_INT << 6) | (BB_FC_DEF << 4) | (int(BB_WIDE_DEF) << 3) | 1   # CR29 del paso 2 del init

# --- Solo para MODO = 'ADRF_REAL' (mediciones de banco) ---
TOL_FREQ_PPM    = 100    # tolerancia total PROPUESTA para frecuencias medidas (oscilador U2 + instrumento).
                         # No es una especificacion: ajustala con las hojas de datos de U2 (DSC1001DL2) y de tu instrumento.
RF_ENTRADA_DBM  = -60    # nivel de la portadora de RF en el barrido de ganancia (sugerencia: no satura ni a 60 dB de ganancia)
F_RF_PRUEBA_HZ  = 161_975_000   # RF de prueba del barrido de ganancia: tiene que caer dentro del filtro de entrada de la placa (FB1, 156 a 162 MHz)
F_BB_MHZ        = 10     # el LO del barrido de ganancia se pone en F_RF_PRUEBA_HZ - F_BB_MHZ: el tono de banda base cae a esa frecuencia
CORRIENTE_EN    = '12V'  # donde se mide la corriente: '12V' = entrada J7 (lo habitual) | '3V3' = riel de 3,3 V del ADRF (tester en serie)

assert MODO in ('ESP32', 'ADRF_REAL'), "MODO debe ser 'ESP32' o 'ADRF_REAL'"
LAB = (MODO == 'ADRF_REAL')
print(f"MODO = {MODO} | INTERACTIVO = {INTERACTIVO} | config = {CONF_ID} | LO = {LO_DEFECTO_HZ/1e6:.6f} MHz | PFD = {PFD_HZ/1e6:.3f} MHz | BSCDIV = {BSCDIV} (0x{BSCDIV:02X})")

MODO = ADRF_REAL | INTERACTIVO = True | config = servidor, crc32 c5c3027c | LO = 151.275000 MHz | PFD = 13.500 MHz | BSCDIV = 56 (0x38)


In [30]:
# ============================================================
# Ayudas: verificaciones, mediciones de banco, bloques de "esperado" y calculo independiente
# ============================================================
RES = []        # (nombre, ok) de las verificaciones automaticas
MANUALES = []   # (descripcion, resultado) -- True/False, o None si no se confirmo
MEDS = []       # mediciones de banco, en orden (diccionarios)
MED_VAL = {}    # clave -> valor medido (float), para los criterios relativos

META = [("fecha_inicio", T_INICIO), ("modo", MODO), ("interactivo", INTERACTIVO), ("servidor", f"{HOST}:{PORT}"),
        ("lo_defecto_hz", LO_DEFECTO_HZ), ("tol_freq_ppm", TOL_FREQ_PPM), ("rf_entrada_dbm", RF_ENTRADA_DBM),
        ("f_bb_mhz", F_BB_MHZ), ("corriente_en", CORRIENTE_EN),
        ("config_origen", CONFIG_ORIGEN), ("conf_id", CONF_ID)]
CSV_CAMPOS = ["hora", "tipo", "etapa", "descripcion", "esperado", "medido", "unidad", "resultado", "detalle"]

def _csv_fila(tipo, descripcion, etapa="", esperado="", medido=None, unidad="", resultado="", detalle=""):
    """Agrega una fila al CSV de resultados. Se escribe al instante: si la prueba se corta, no se pierde lo medido."""
    nuevo = not os.path.exists(ARCHIVO_CSV)
    os.makedirs(DIR_RESULTADOS, exist_ok=True)
    medido = "" if medido is None else (str(medido).replace(".", ",") if isinstance(medido, float) else medido)
    with open(ARCHIVO_CSV, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f, delimiter=";")            # Excel en espanol: separador ';' y coma decimal
        if nuevo:
            w.writerow(CSV_CAMPOS)
            for clave, valor in META:
                w.writerow([time.strftime("%H:%M:%S"), "meta", "", clave, "", "", "", "", valor])
        w.writerow([time.strftime("%H:%M:%S"), tipo, etapa, descripcion, esperado, medido, unidad, resultado, detalle])

def chk(nombre, cond, detalle=""):
    RES.append((nombre, bool(cond)))
    _csv_fila("auto", nombre, resultado="OK" if cond else "FALLO", detalle=detalle)
    return rpio.veredicto(nombre, bool(cond), detalle)

def manual(desc):
    print(f"  [MANUAL] {desc}")
    res = None
    if INTERACTIVO:
        r = input("    Se cumple? (s = si, n = no, Enter = omitir): ").strip().lower()
        res = True if r == "s" else (False if r == "n" else None)
    MANUALES.append((desc, res))
    _csv_fila("manual", desc, resultado={True: "CUMPLIDA", False: "NO CUMPLIDA", None: "SIN CONFIRMAR"}[res])
    return res

def vgain_seguro_raw():
    """Codigo del DAC de ganancia minima: 0 con pendiente positiva; 8191 (el maximo) con la negativa."""
    return 8191 if VGA_POL else 0

def iniciar_adrf():
    """Init segun CONFIG_ORIGEN: 'servidor' -> con el .conf del servidor (adrf_init); 'cliente' -> con CONF (adrf_init_cfg).
    Con VGAIN_POR_DAC, el servidor (arrancado con -d) aplica VGAIN el solo en cada escritura del driver: no hace falta nada aca."""
    return rpio.adrf_init_cfg(sock, CONF) if CONFIG_ORIGEN == 'cliente' else rpio.adrf_init(sock)

MSG_INIT = "ADRF: init (configuracion del cliente) -> 0" if CONFIG_ORIGEN == 'cliente' else "ADRF: init -> 0"

def abortar(msg):
    raise RuntimeError(msg)

def rp(*lineas):
    print("  ESPERADO en la consola del servidor (Red Pitaya):")
    for l in lineas:
        print(f"    RP    > {l}")

def esp(*lineas, laxas=()):
    if MODO != "ESP32":
        return
    print("  ESPERADO en el Serial del ESP32:")
    for l in lineas:
        print(f"    ESP32 > {l}")
    for l in laxas:
        print(f"    ESP32 ~ {l}")

# ---- Mediciones de banco (solo MODO = 'ADRF_REAL') ----
def _leer_valor(prompt):
    for _ in range(3):
        txt = input(prompt).strip()
        if txt == "":
            return None, "OMITIDA"
        if txt.lower() in ("n", "no"):
            return None, "NO MEDIDA"
        try:
            return float(txt.replace(",", ".")), None
        except ValueError:
            print("    No entendi el valor: escribir un numero (por ejemplo 3,31). Enter omite la medicion.")
    return None, "OMITIDA"

def medir(clave, etapa, magnitud, unidad, instrumento, como="", esperado=None, tol=None, tol_ppm=None,
          rango=None, rel=None, tipico=None, orientativo=False):
    """Una medicion de banco. Criterio (solo uno): esperado +/- tol (o tol_ppm), rango=(min, max) con None
    para un lado abierto, o rel=(clave_previa, 'dif', valor, tol) respecto de una medicion anterior.
    Sin criterio, solo se anota. Con INTERACTIVO=False solo imprime el plan."""
    if not LAB:
        return None
    lo = hi = objetivo = None
    texto = "solo anotar (sin criterio de aprobacion)"
    if rel is not None:
        previa, tipo, val, tol_r = rel
        pv = MED_VAL.get(previa)
        if pv is None and INTERACTIVO:
            texto = f"relativo a la medicion '{previa}', que no se midio: solo anotar"
        else:
            base = pv if pv is not None else 0.0
            objetivo = base + val
            lo, hi = objetivo - tol_r, objetivo + tol_r
            ref = f"la medicion anterior ({pv:g})" if pv is not None else "la medicion anterior"
            texto = f"{val:+g} {unidad} respecto de {ref}, con tolerancia +/-{tol_r:g}"
            if pv is not None:
                texto += f"  ->  entre {lo:g} y {hi:g} {unidad}"
    elif esperado is not None:
        t = abs(esperado) * tol_ppm * 1e-6 if tol_ppm is not None else (tol or 0.0)
        objetivo, lo, hi = esperado, esperado - t, esperado + t
        texto = f"{esperado:g} {unidad} +/- {t:g}" + (f" ({tol_ppm:g} ppm)" if tol_ppm is not None else "")
    elif rango is not None:
        lo, hi = rango
        objetivo = (tipico if tipico is not None else
                    (lo + hi) / 2 if (lo is not None and hi is not None) else
                    hi * 0.5 if lo is None else lo * 1.5)
        texto = (f"entre {lo:g} y {hi:g} {unidad}" if (lo is not None and hi is not None) else
                 f"como maximo {hi:g} {unidad}" if lo is None else f"como minimo {lo:g} {unidad}")
    print(f"  [MEDIR] {etapa}")
    print(f"          {magnitud} [{unidad}] con {instrumento}")
    if como:
        print(f"          como: {como}")
    print(f"          esperado: {texto}" + ("  (criterio orientativo propuesto, no es especificacion)" if orientativo else ""))
    if tipico is not None:
        print(f"          valor tipico segun la hoja de datos: {tipico:g} {unidad}")
    reg = dict(clave=clave, etapa=etapa, magnitud=magnitud, unidad=unidad, instrumento=instrumento,
               criterio=texto + (" (orientativo)" if orientativo else ""), medido=None,
               veredicto="SIN MEDIR", comentario="", hora=time.strftime("%H:%M:%S"))
    MEDS.append(reg)
    try:
        if not INTERACTIVO:
            return None
        medido, motivo = _leer_valor(f"    Valor medido [{unidad}] (Enter = omitir, n = no pude medir): ")
        if medido is None:
            reg["veredicto"] = motivo
            print(f"    -> {motivo}")
            return None
        reg["medido"] = medido
        MED_VAL[clave] = medido
        if lo is None and hi is None:
            reg["veredicto"] = "ANOTADA"
            print(f"    -> anotado: {medido:g} {unidad}")
            return medido
        ok = (lo is None or medido >= lo) and (hi is None or medido <= hi)
        reg["veredicto"] = "OK" if ok else "FUERA"
        rpio.veredicto(etapa, ok, f"medido = {medido:g} {unidad}; esperado: {texto}")
        if not ok:
            reg["comentario"] = input("    Comentario sobre la desviacion (opcional, Enter para omitir): ").strip()
        return medido
    finally:
        _csv_fila("medicion", reg["magnitud"], etapa=reg["etapa"], esperado=reg["criterio"], medido=reg["medido"],
                  unidad=reg["unidad"], resultado=reg["veredicto"],
                  detalle=" | ".join(x for x in (reg["instrumento"], reg["comentario"]) if x))

def medir_corriente(clave, etapa, como="", criterio=True):
    """La hoja de datos da 350 mA tipicos y 440 mA maximos en el riel de 3,3 V del ADRF. En la entrada de 12 V
    la corriente es menor (la placa tiene un conversor y un regulador lineal), asi que alli solo se anota."""
    if CORRIENTE_EN == "3V3":
        return medir(clave, etapa, "corriente del riel de 3,3 V del ADRF", "mA", "tester en serie con el riel de 3,3 V",
                     como=como, rango=(None, 440) if criterio else None, tipico=350 if criterio else None)
    est_ma = 3.3 * 350 / (12 * 0.8)
    return medir(clave, etapa, "corriente de entrada de 12 V (J7)", "mA", "fuente de laboratorio",
                 como=(como + "; " if como else "") + f"solo se anota: con el ADRF en 350 mA a 3,3 V (tipico) la entrada de 12 V deberia dar ~{est_ma:.0f} mA "
                      "(estimacion con 80 % de rendimiento) mas lo que consuman los demas circuitos de la placa")


# ---- Calculo independiente de la hoja de datos (no usa el codigo C) ----
def plan_lo(lo_hz, pfd=PFD_HZ):
    """RFDIV (Tabla 6), N = 2^(RFDIV+1) * LO / PFD (Ec. 4) y N = INT + FRAC/2^25 (Ec. 5)."""
    d = 0 if lo_hz >= 500_000_000 else 1 if lo_hz >= 250_000_000 else 2 if lo_hz >= 125_000_000 else 3
    q25 = (((lo_hz << (d + 1)) << 25) + pfd // 2) // pfd
    sh = 25 + d + 1
    lo_act = (pfd * q25 + (1 << (sh - 1))) >> sh
    return dict(lo_hz=lo_hz, rfdiv=d, n_int=q25 >> 25, n_frac=q25 & ((1 << 25) - 1),
                lo_actual=lo_act, vco_mhz=(lo_act * 4 * (1 << d)) // 1_000_000)

def vgain_mv(gain_mdb):
    """VGAIN = 25 mV/dB. Con pendiente positiva (vga_pendiente = 0) 0 V = 0 dB; con negativa se invierte (1500 - mV)."""
    mv = (gain_mdb * 25 + 500) // 1000
    return 1500 - mv if VGA_POL else mv

def wr(addr, val):
    return f"WRITE CR{addr:<2d} (0x{addr:02X}) = 0x{val:02X}"

def txt_acq(p, autocal_on=True):
    cal_us = BSCDIV * 24 * 1_000_000 // PFD_HZ
    lo = p['lo_actual']
    return (f"ACQ   (CR0) PFD={PFD_HZ // 1_000_000}.{(PFD_HZ % 1_000_000) // 1000:03d} MHz  "
            f"N={p['n_int']}+{p['n_frac']}/2^25  RFDIV={p['rfdiv']}  "
            f"LO={lo // 1_000_000}.{lo % 1_000_000:06d} MHz  VCO={p['vco_mhz']} MHz  "
            f"autocal {'on' if autocal_on else 'OFF'} ({cal_us} us)")

def regs_lo(p):
    """Escrituras de un cambio de LO, en el orden del driver."""
    return [(28, 0x08 | p['rfdiv']), (7, (MUXOUT << 4) | ((p['n_int'] >> 8) & 0x0F)), (6, p['n_int'] & 0xFF),
            (3, (p['n_frac'] >> 24) & 1), (2, (p['n_frac'] >> 16) & 0xFF),
            (1, (p['n_frac'] >> 8) & 0xFF), (0, p['n_frac'] & 0xFF)]

def lineas_lo_esp32(p, autocal_on=True, cr24=None):
    out = []
    if cr24 is not None:
        out.append(wr(24, cr24))
    out += [wr(a, v) for a, v in regs_lo(p)]
    out += [txt_acq(p, autocal_on), "LDET: engancha en ~260 us"]
    return out

def regs_init(p):
    """Los 31 registros de la secuencia de la hoja de datos, en orden (pasos 1 a 31)."""
    cr10 = (DIV2 << 6) | (DOBLADOR << 5) | (0 if RDIV == 32 else RDIV & 0x1F)
    return [(30, 0x00), (29, CR29_CONF), (28, 0x08 | p['rfdiv']), (27, (LOMON_EN_DEF << 2) | LOMON_PWR_DEF), (26, 0x00), (25, BSCDIV),
            (24, 0x38), (23, 0x70), (22, 0), (21, 0), (20, 0), (19, 0), (18, 0x60), (17, 0), (16, 0),
            (15, 0), (14, 0), (13, 0x08), (12, 0x18), (11, 0), (10, cr10), (9, CP_DEFECTO << 4), (8, 0),
            (7, (MUXOUT << 4) | ((p['n_int'] >> 8) & 0x0F)), (6, p['n_int'] & 0xFF), (5, RDIV_EN << 4), (4, 0x01),
            (3, (p['n_frac'] >> 24) & 1), (2, (p['n_frac'] >> 16) & 0xFF),
            (1, (p['n_frac'] >> 8) & 0xFF), (0, p['n_frac'] & 0xFF)]

def lineas_init_esp32(p, reinit=False):
    out = (["CS pulso extra tras el lock (1 byte 0x00): el chip lo ignora"] * 3 if reinit
           else ["CS pulso 1/3", "CS pulso 2/3", "CS pulso 3/3"])
    for i, (a, v) in enumerate(regs_init(p)):
        out.append(wr(a, v))
        if i == 0:
            out.append("INIT: comienzo de secuencia detectado (CR30 con VGA apagado)")
        if a == 0:
            out += [txt_acq(p, True), "LDET: engancha en ~260 us"]
    out += [wr(30, (VGA_POL << 2) | 1), "INIT: secuencia COMPLETA y conforme a la hoja de datos"]
    return out

def estado_esperado(lo_hz=LO_DEFECTO_HZ, gain_mdb=GAIN_DEFECTO_MDB, bb_wideband=BB_WIDE_DEF, bb_fc=BB_FC_DEF,
                    cp=CP_DEFECTO, autocal_off=False):
    p = plan_lo(lo_hz)
    return {'initialized': True, 'lo_hz': lo_hz, 'lo_actual_hz': p['lo_actual'], 'rfdiv': p['rfdiv'],
            'n_int': p['n_int'], 'n_frac': p['n_frac'], 'gain_mdb': gain_mdb,
            'vgain_mv': vgain_mv(gain_mdb), 'bb_wideband': bb_wideband, 'bb_fc': bb_fc,
            'cp_code': cp, 'autocal_off': autocal_off}

def comparar_estado(est, esperado, campos=None):
    for campo in (campos or esperado.keys()):
        chk(f"estado.{campo}", est[campo] == esperado[campo],
            f"esperado={esperado[campo]}  real={est[campo]}")

## 0. Lista previa y conexión

Esta lista se hace **antes** de mandar ningún comando por SPI. En `ADRF_REAL` incluye las mediciones de alimentación, referencia y `CS`.

In [31]:
print("Consola del servidor (Red Pitaya), al arrancar:")
rp("ADRF: /dev/spidev1.0 abierto (modo 0, 8 bits, 1000000 Hz)", f"Servidor ADRF escuchando en puerto {PORT}")
print("    Si aparece 'ADRF: no se pudo abrir ...' el servidor termina solo (sin SPI no hay nada que probar): revisar que exista /dev/spidev1.0.")
manual("La consola del servidor muestra las dos lineas de arriba y ningun mensaje de error")
print()

socket.setdefaulttimeout(5.0)
try:
    sock = rpio.conectar_comandos(HOST, PORT)
except OSError as e:
    chk("conexion TCP al servidor", False, f"{HOST}:{PORT} -> {e}")
    abortar("No hay conexion con el servidor: revisar el cable Ethernet, la IP (HOST), el puerto (PORT) y que adrf_server este corriendo")
finally:
    socket.setdefaulttimeout(None)
sock.settimeout(10.0)
chk("conexion TCP al servidor", True, f"{HOST}:{PORT}")


Consola del servidor (Red Pitaya), al arrancar:
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: /dev/spidev1.0 abierto (modo 0, 8 bits, 1000000 Hz)
    RP    > Servidor ADRF escuchando en puerto 1005
    Si aparece 'ADRF: no se pudo abrir ...' el servidor termina solo (sin SPI no hay nada que probar): revisar que exista /dev/spidev1.0.
  [MANUAL] La consola del servidor muestra las dos lineas de arriba y ningun mensaje de error

  [OK] conexion TCP al servidor: 192.168.1.100:1005


True

In [4]:
print(f"=== Lista previa ({MODO}) ===")
if MODO == 'ESP32':
    manual("Sketch cargado; el Serial Monitor (115200) muestra 'Emulador ADRF6850' y 'Arranca en modo I2C'")
    manual("Cableado: E2 pin 3 -> GPIO23, pin 4 <- GPIO19, pin 5 -> GPIO18, pin 6 -> GPIO5, GND comun")
else:
    manual("Solo cables individuales para CLK, SDI, SDO, CS y GND hacia el Red Pitaya; J3 pines 1-2 (+3,3 V de la placa) SIN conectar")
    medir("vcc", "Alimentacion del ADRF (antes de programar)", "tension en los VCC del U9", "V", "multimetro",
          como="medir en los pines 1, 13, 14, 15, 16, 31, 36, 42 y 49 del U9 (al menos 3) y cargar el MENOR",
          rango=(3.15, 3.45), tipico=3.3)
    medir_corriente("i_pre", "Alimentacion del ADRF (antes de programar)", como="anotarla; la hoja de datos no da valor antes de programar", criterio=False)
    medir("refin_f", "Referencia", "frecuencia de la referencia (J8, nodo de REFIN)", "MHz", "contador o osciloscopio con sonda de alta impedancia",
          como="segun la netlist, J8 es una derivacion del nodo del oscilador U2: no cargarlo con 50 ohm, que arrastraria al oscilador",
          esperado=REFIN_HZ / 1e6, tol_ppm=TOL_FREQ_PPM, orientativo=True)
    medir("refin_v", "Referencia", "amplitud de la referencia (J8, nodo de REFIN)", "Vpp", "osciloscopio con sonda de alta impedancia",
          como="el oscilador es CMOS de 3,3 V y llega a REFIN acoplado en AC por C41 y C42 en serie", rango=(0.4, 3.45))
    medir("refin_pin_dc", "Referencia", "continua en el pad de C42 del lado de REFIN (pin 17), con el chip alimentado", "V", "tester",
          como="con el chip alimentado la entrada tiene una polarizacion interna: 0 V exacto sugiere el pin 17 sin conectar (la hoja de datos no da el valor)")
    medir("refin_pin_v", "Referencia", "amplitud en el pad de C42 del lado de REFIN (pin 17)", "Vpp", "osciloscopio con sonda de alta impedancia",
          como="acople dc y masa corta. La hoja de datos pide de 0,4 Vpp a VCC; un exceso de sobrepico indica falta de amortiguamiento en R28",
          rango=(0.4, 3.3))
    medir("cs_alto0", "Pull-up de CS (sin trafico SPI)", "nivel de CS en reposo (pin 27)", "V", "multimetro",
          como="con el SPI todavia desconectado; debe estar en alto por R10", rango=(1.4, 3.45), tipico=3.3)
    medir("r10", "Pull-up de CS", "valor de R10", "ohm", "multimetro",
          como="con la placa APAGADA y desconectada; en circuito puede leer menos. La netlist dice 330 ohm: si no era la intencion, ver la nota de R10",
          esperado=330, tol=17, orientativo=True)
    manual("LED D3 (LDET) apagado antes de programar (se espera: detector de lock deshabilitado por defecto; la hoja de datos no da el nivel de LDET en ese caso, por eso se mira)")
    if VGAIN_POR_DAC and VERIFICAR_DAC:
        manual("adrf_server arrancado con -d (sin -F ni -O todavia); OUT2 de la Red Pitaya SIN conectar a J2 (VGAIN): se conecta en la seccion 0b")
    elif VGAIN_POR_DAC:
        manual("adrf_server arrancado con -d -F <mV> -O <mV> (la calibracion de la seccion 0b) y OUT2 conectado a J2 (VGAIN)")
    else:
        manual("VGAIN (J2): fuente externa de 0 a 1,5 V lista y en 0 V (ganancia minima) antes de correr el init; el pin es de 20 kohm: flotante = ganancia indefinida")
    manual("J1 (entrada de RF) con una carga de 50 ohm y sin generador conectado")
    manual("Instrumentos listos: analizador de espectro (LOMON en J4, banda base en J5 = I y J6 = Q), osciloscopio (senales SPI) y tester")
print()


=== Lista previa (ADRF_REAL) ===
  [MANUAL] Solo cables individuales para CLK, SDI, SDO, CS y GND hacia el Red Pitaya; J3 pines 1-2 (+3,3 V de la placa) SIN conectar
  [MEDIR] Alimentacion del ADRF (antes de programar)
          tension en los VCC del U9 [V] con multimetro
          como: medir en los pines 1, 13, 14, 15, 16, 31, 36, 42 y 49 del U9 (al menos 3) y cargar el MENOR
          esperado: entre 3.15 y 3.45 V
          valor tipico segun la hoja de datos: 3.3 V
    -> OMITIDA
  [MEDIR] Alimentacion del ADRF (antes de programar)
          corriente de entrada de 12 V (J7) [mA] con fuente de laboratorio
          como: anotarla; la hoja de datos no da valor antes de programar; solo se anota: con el ADRF en 350 mA a 3,3 V (tipico) la entrada de 12 V deberia dar ~120 mA (estimacion con 80 % de rendimiento) mas lo que consuman los demas circuitos de la placa
          esperado: solo anotar (sin criterio de aprobacion)
    -> OMITIDA
  [MEDIR] Referencia
          frecuencia de la r

## 0b. DAC de VGAIN: calibración antes de conectarlo (solo `ADRF_REAL`)

`VGAIN` lo aplica el DAC de la Red Pitaya por registro (`adrf_server` arrancado con `-d`: ver `adrf6850/LEEME.txt`). **Con `OUT2` todavía sin conectar a `J2`**, esta sección sube el código de a poco con `VGAIN_RAW` (nunca más de lo medido en el paso anterior), mide la tensión real y calcula el fondo de escala y el offset del DAC.

Al terminar pide **reiniciar `adrf_server` con `-F` y `-O`** (los valores que imprime) para que el servidor aplique `VGAIN` con esa calibración, y recién ahí conectar `OUT2` a `J2`. Con `VERIFICAR_DAC = False` (ya calibrado y conectado) esta sección solo dejar `VGAIN` en la ganancia mínima antes de continuar.

In [ ]:
# ============================================================
# 0b. DAC de VGAIN: calibracion con el osciloscopio, con OUT2 todavia desconectado de J2
# ============================================================
if VGAIN_POR_DAC:
    print("=== 0b. DAC de VGAIN ===")
    g = rpio.adrf_get_vgain(sock)
    if not g['activo']:
        abortar("El servidor no tiene el DAC activado: arrancar adrf_server_dacreg con -d (y, una vez calibrado, con -F y -O)")

    def poner_codigo(c):
        if not rpio.adrf_vgain_raw(sock, c):
            abortar(f"El servidor rechazo el codigo {c} (VGAIN_RAW)")

    if VERIFICAR_DAC and INTERACTIVO:
        print("  OUT2 todavia sin conectar a J2.")
        LIMITE_V = 1.6                # si se mide mas que esto (o menos que -0,05 V), la seccion se corta y el DAC queda en 0

        manual("Osciloscopio (acople dc, 1 Mohm) en OUT2; OUT2 sin conectar a J2 ni a ninguna otra cosa")
        poner_codigo(0)
        print("  DAC en el codigo 0: VGAIN = 0 V.")
        # Escalera: al principio se supone 1 V de fondo de escala; despues cada codigo sale de lo medido, para no pasarse
        fs_mv = 1000.0
        codigos, tensiones = [], []
        v0 = medir("dac_0", "Escalera del DAC", "tension en OUT2 con el codigo 0", "V", "osciloscopio, acople dc",
                   como="debe ser casi 0 V: con pendiente positiva es la ganancia minima", rango=(-0.05, 0.05))
        if v0 is not None:
            codigos.append(0); tensiones.append(v0)
        for objetivo in (0, 250, 500, 1000, 1250, 1500):
            c = min(round(objetivo / fs_mv * rpio.DAC_CODIGO_MAX), rpio.DAC_CODIGO_MAX)
            if c in codigos:
                print(f"  {objetivo} mV: el codigo maximo ({c}) ya se midio; el DAC no da para mas. Se corta la escalera aca.")
                break
            poner_codigo(c)
            v = medir(f"dac_{objetivo}", "Escalera del DAC", f"tension en OUT2 con el codigo {c} (se pidieron {objetivo} mV)", "V",
                      "osciloscopio, acople dc", como="valor medio de la continua")
            if v is None:
                continue
            if v > LIMITE_V or v < -0.05:
                poner_codigo(0)
                abortar(f"Se midieron {v:.3f} V en OUT2 (limite {LIMITE_V} V): no conectar a J2. DAC en 0.")
            codigos.append(c); tensiones.append(v)
            if c > 0:
                fs_mv = max(v / c * rpio.DAC_CODIGO_MAX * 1000.0, 1.0)
        if len(set(codigos)) >= 2:
            fs, off = rpio.ajustar_dac(codigos, tensiones)             # en voltios
            residuo = np.array(tensiones) - (off + fs * np.array(codigos) / rpio.DAC_CODIGO_MAX)
            print(f"  Calibracion: fondo de escala = {fs * 1e3:.1f} mV, offset = {off * 1e3:.2f} mV, "
                  f"mayor desvio de la recta = {np.abs(residuo).max() * 1e3:.2f} mV")
            print(f"  Reiniciar adrf_server_dacreg con -F {fs * 1e3:.0f} -O {off * 1e3:.1f} para que el servidor aplique VGAIN con esta calibracion.")
            chk("0b el DAC sigue una recta (mayor desvio < 5 mV)", np.abs(residuo).max() < 0.005, f"{np.abs(residuo).max() * 1e3:.2f} mV")
        poner_codigo(0)
        manual("OUT2 conectado a J2 (VGAIN). Recomendado: 100 ohm en serie y un diodo Schottky de masa a VGAIN (anodo a masa) contra tensiones negativas")
        abortar("Reiniciar adrf_server_dacreg con -F y -O (arriba), reconectar y poner VERIFICAR_DAC = False para continuar con la calibracion aplicada.")
    poner_codigo(vgain_seguro_raw())
    print("  VGAIN en la ganancia minima.")

=== 0b. DAC de VGAIN ===
  OUT2 todavia sin conectar a J2.
  [MANUAL] Osciloscopio (acople dc, 1 Mohm) en OUT2; OUT2 sin conectar a J2 ni a ninguna otra cosa
  DAC en el codigo 0: VGAIN = 0 V.
  [MEDIR] Escalera del DAC
          tension en OUT2 con el codigo 0 [V] con osciloscopio, acople dc
          como: debe ser casi 0 V: con pendiente positiva es la ganancia minima
          esperado: entre -0.05 y 0.05 V
    -> OMITIDA
  [MEDIR] Escalera del DAC
          tension en OUT2 con el codigo 8191 (se pidieron 1500 mV) [V] con osciloscopio, acople dc
          como: valor medio de la continua
          esperado: solo anotar (sin criterio de aprobacion)
    -> OMITIDA
  [MEDIR] Escalera del DAC
          tension en OUT2 con el codigo 8191 (se pidieron 1250 mV) [V] con osciloscopio, acople dc
          como: valor medio de la continua
          esperado: solo anotar (sin criterio de aprobacion)
    -> OMITIDA
  [MEDIR] Escalera del DAC
          tension en OUT2 con el codigo 8191 (se pidi

RuntimeError: Reiniciar adrf_server_dacreg con -F y -O (arriba), reconectar y poner VERIFICAR_DAC = False para continuar con la calibracion aplicada.

## 1. Inicialización completa

Primera prueba real del cambio `I2C→SPI` (3 pulsos de CS) y de la secuencia de `33` pasos de la hoja de datos. El `ack` que devuelve `adrf_init` **no** confirma el enganche del PLL ni que el chip haya recibido algo.

In [32]:
p0 = plan_lo(LO_DEFECTO_HZ)
print(f"=== 1. adrf_init con la configuracion del .conf: LO={LO_DEFECTO_HZ/1e6:.6f} MHz -> RFDIV={p0['rfdiv']}, N={p0['n_int']}+{p0['n_frac']}/2^25 ===")
rp(f"ADRF: VGAIN externo (J2): aplicar {vgain_mv(0)} mV",
   f"ADRF: VGAIN externo (J2): aplicar {vgain_mv(GAIN_DEFECTO_MDB)} mV",
   MSG_INIT)
esp(*lineas_init_esp32(p0),
    laxas=["el 3er pulso de CS termina con '-> SPI ENGANCHADO'", "LDET -> 1", "(la linea 'LDET -> 1' puede aparecer despues de 'INIT: secuencia COMPLETA...')"])

ok_init = iniciar_adrf()
chk("adrf_init: ack True (el driver corrio; NO prueba que el chip recibio algo)", ok_init, f"ack={ok_init}")
_v = rpio.adrf_get_verify(sock)
if _v['retries'] or _v['failures']:
    print(f"  Enlace SPI: {_v['writes']} escrituras verificadas, {_v['retries']} reintentos, {_v['failures']} fallas")
if not ok_init:
    _det = (f" La verificacion fallo en CR{_v['fail_addr']}: escrito 0x{_v['fail_wrote']:02X}, leido 0x{_v['fail_read']:02X}, aun despues de los reintentos."
            if _v['failures'] else "")
    abortar("adrf_init devolvio False: el driver devolvio un error." + _det + " (ver la consola del servidor)")

if MODO == 'ESP32':
    manual("El Serial del ESP32 muestra la secuencia esperada de arriba, termina en 'INIT: secuencia COMPLETA y conforme' y el LED azul (D3) queda encendido")
else:
    manual("LED D3 encendido pocos instantes despues del init (el detector de lock solo se habilita por SPI: es la mejor evidencia de que llego, si D3 estaba apagado antes)")
    medir_corriente("i_init", "Despues del init", como="con todos los bloques encendidos (PLL, demodulador, VGA)")
    medir("vcc_init", "Despues del init", "tension en los VCC del U9 con el ADRF funcionando", "V", "multimetro",
          rango=(3.15, 3.45), tipico=3.3)

=== 1. adrf_init con la configuracion del .conf: LO=151.275000 MHz -> RFDIV=2, N=89+21623967/2^25 ===
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: VGAIN externo (J2): aplicar 0 mV
    RP    > ADRF: VGAIN externo (J2): aplicar 1000 mV
    RP    > ADRF: init -> 0
  [OK] adrf_init: ack True (el driver corrio; NO prueba que el chip recibio algo): ack=True
  [MANUAL] LED D3 encendido pocos instantes despues del init (el detector de lock solo se habilita por SPI: es la mejor evidencia de que llego, si D3 estaba apagado antes)
  [MEDIR] Despues del init
          corriente de entrada de 12 V (J7) [mA] con fuente de laboratorio
          como: con todos los bloques encendidos (PLL, demodulador, VGA); solo se anota: con el ADRF en 350 mA a 3,3 V (tipico) la entrada de 12 V deberia dar ~120 mA (estimacion con 80 % de rendimiento) mas lo que consuman los demas circuitos de la placa
          esperado: solo anotar (sin criterio de aprobacion)
    -> OMITIDA
  [MEDIR] Despu

## 2. Estado post-init

Valores de la configuración (la del `.conf` del servidor o la del cliente, según `CONFIG_ORIGEN`), contrastados con el cálculo independiente de este notebook. Recordá que es el estado interno del driver, no una lectura del chip.

In [42]:
print("=== 2. estado post-init ===")
est = rpio.adrf_get_state(sock)
comparar_estado(est, estado_esperado())

=== 2. estado post-init ===
  [OK] estado.initialized: esperado=True  real=True
  [OK] estado.lo_hz: esperado=330000000  real=330000000
  [OK] estado.lo_actual_hz: esperado=330000000  real=330000000
  [OK] estado.rfdiv: esperado=1  real=1
  [OK] estado.n_int: esperado=48  real=48
  [OK] estado.n_frac: esperado=29826162  real=29826162
  [OK] estado.gain_mdb: esperado=20000  real=20000
  [OK] estado.vgain_mv: esperado=500  real=500
  [OK] estado.bb_wideband: esperado=False  real=False
  [OK] estado.bb_fc: esperado=0  real=0
  [OK] estado.cp_code: esperado=7  real=7
  [OK] estado.autocal_off: esperado=False  real=False


## 2a. Lectura de registros por SDO

Lee de vuelta, por `SDO`, registros que el init acaba de escribir y compara **solo los bits documentados** con los valores de la secuencia de la hoja de datos. Es la primera verificación automática que puede detectar un enlace SPI muerto: si lo escrito por `SDI` vuelve por `SDO`, el chip recibió los comandos y la vía de retorno funciona.

Los registros con doble buffer (frecuencia, divisor, bomba de carga, referencia) solo se anotan: la hoja de datos no dice qué devuelve su lectura (la sección 2d los compara todos). El código de revisión (`CR33`) no tiene valor documentado: en el ADRF real solo se exige que no sea `0x00` ni `0xFF`.

In [ ]:
print("=== 2a. lectura de registros por SDO: prueba del enlace de ida y vuelta ===")
# Lo escrito en el init sale de la secuencia de la hoja de datos (regs_init). Aca se lee de vuelta por SDO y se compara.
# Solo se comparan los bits DOCUMENTADOS de cada registro (mascara): los reservados pueden no guardarse en el chip.
ESCRITO = dict(regs_init(p0)); ESCRITO[30] = (VGA_POL << 2) | 1   # CR30 termina asi (paso 33 de la secuencia)
LECTURAS = [   # (registro, mascara de bits documentados, valor esperado en esos bits, descripcion)
    (25, 0xFF, ESCRITO[25] & 0xFF, "temporizador de autocalibracion"),
    (29, 0x79, ESCRITO[29] & 0x79, "demodulador encendido, VOCM y filtro de banda base del .conf"),
    (30, 0x05, ESCRITO[30] & 0x05, "VGA encendido y pendiente del .conf"),
    (24, 0x01, ESCRITO[24] & 0x01, "autocalibracion habilitada"),
    (23, 0x1C, ESCRITO[23] & 0x1C, "detector de lock habilitado, 3072 pulsos, precision gruesa"),
    (12, 0x04, ESCRITO[12] & 0x04, "PLL encendido"),
    (27, 0x07, ESCRITO[27] & 0x07, "LOMON apagado"),
]
SOLO_ANOTAR = [(9, "bomba de carga (doble buffer)"), (10, "referencia (doble buffer)"), (28, "RFDIV (doble buffer)")]
ORDEN_LECTURA = [a for a, *_ in LECTURAS] + [a for a, _ in SOLO_ANOTAR]

if MODO == 'ESP32':
    rp(*[f"ADRF: read_reg(CR{a}) -> 0x{ESCRITO[a]:02X}" for a in ORDEN_LECTURA])
    esp(*[l for a in ORDEN_LECTURA for l in (f"READ-SELECT CR{a}  (0xD4 + addr)",
                                              f"READ  CR{a} -> 0x{ESCRITO[a]:02X}  (0xD5 + dato en el mismo frame)")],
        laxas=["READ-SELECT CR33 y READ CR33 (el codigo de revision que devuelve el emulador)"])
else:
    print("  Se leen de vuelta CR25, CR29, CR30, CR24, CR23, CR12 y CR27 (se comparan los bits documentados),")
    print("  CR9, CR10 y CR28 (solo se anotan) y CR33 (revision).")

def leer(a):
    try:
        return rpio.adrf_read_reg(sock, a)
    except (RuntimeError, ValueError, ConnectionError, OSError) as e:
        chk(f"lectura de CR{a} por SDO", False, str(e))
        abortar(f"No se pudo leer CR{a}: {e}")

valores, fallos = {}, 0
for a, mascara, esperado, que in LECTURAS:
    v = leer(a); valores[a] = v
    ok = (v & mascara) == esperado
    fallos += (not ok)
    chk(f"readback CR{a} ({que}; bits documentados 0x{mascara:02X})", ok, f"leido 0x{v:02X} -> 0x{v & mascara:02X}; esperado 0x{esperado:02X}")
for a, que in SOLO_ANOTAR:
    v = leer(a)
    print(f"  [ANOTADA] CR{a} {que}: leido 0x{v:02X} (escrito 0x{ESCRITO[a]:02X})")
    _csv_fila("lectura", f"CR{a} {que}", esperado=f"escrito 0x{ESCRITO[a]:02X}", medido=f"0x{v:02X}", resultado="ANOTADA")
rev = leer(33)
if LAB:
    chk("readback CR33 (codigo de revision): distinto de 0x00 y 0xFF (la hoja de datos no da su valor)", rev not in (0x00, 0xFF), f"leido 0x{rev:02X}")
else:
    print(f"  [ANOTADA] CR33 (codigo de revision que devuelve el emulador): 0x{rev:02X}")
    _csv_fila("lectura", "CR33 codigo de revision (emulador)", medido=f"0x{rev:02X}", resultado="ANOTADA")

# Las lecturas que esperan un valor distinto de cero son las que de verdad prueban el enlace: un SDO muerto o trabado en 0
# (chip sin alimentar, MISO a masa) pasa de casualidad las que esperan 0x00.
informativas = [a for a, _, esperado, _ in LECTURAS if esperado != 0]
malas = [a for a, mascara, esperado, _ in LECTURAS if (valores[a] & mascara) != esperado]
if all(a in malas for a in informativas):
    crudos = {valores[a] for a in informativas}
    if crudos == {0x00}:
        causa = "las lecturas dieron 0x00: SDO sin conectar o sin contestar, o el chip sin alimentar o sin enganchar en SPI"
    elif crudos == {0xFF}:
        causa = "las lecturas dieron 0xFF: SDO flotante o en alto (cable de SDO desconectado)"
    else:
        causa = "valores distintos de los escritos: las escrituras no estan llegando (CS, CLK, SDI, modo I2C) o falta la masa comun"
    print(f"  *** El enlace SPI NO responde como se espera ({causa}).")
    print("  *** Parar y revisar el cableado y la alimentacion antes de seguir con las demas secciones.")
elif malas:
    print(f"  *** {len(malas)} de {len(LECTURAS)} lecturas no coinciden con lo escrito (CR{', CR'.join(str(a) for a in malas)}): ver los [FALLO] de arriba.")
else:
    print("  Enlace SPI verificado en las dos direcciones: lo escrito por SDI se lee de vuelta por SDO.")


## 2b. Niveles lógicos de las señales SPI (solo `ADRF_REAL`, osciloscopio)

Comprueba los niveles de las señales SPI **en los pines del U9** contra la tabla de niveles lógicos de la hoja de datos. Es la medición que importa por `R10`: con un pull-up de 330 Ω, el Red Pitaya tiene que bajar `CS` por debajo de 0,6 V mientras absorbe unos 10 mA. Cada `adrf_init` manda ~35 frames en pocos milisegundos: configurá el disparo y el notebook enviará un init cuando lo pidas (re-ejecutá la celda para otra captura).

In [43]:
if LAB:
    print("=== 2b. niveles logicos de las senales SPI (osciloscopio) ===")
    print("  Sondas en CS (pin 27), CLK (pin 30) y SDI (pin 29) del U9, con masa corta. Disparo: flanco de bajada de CS.")
    if INTERACTIVO:
        input("    Configurar el osciloscopio y presionar Enter para enviar un adrf_init... ")
        iniciar_adrf()
    medir("cs_bajo", "Senales SPI", "nivel BAJO de CS durante una transaccion (pin 27)", "V", "osciloscopio",
          como="es el nivel que fija R10 (330 ohm) junto con la capacidad de corriente del pin del Red Pitaya", rango=(None, 0.6))
    medir("cs_alto", "Senales SPI", "nivel ALTO de CS entre transacciones (pin 27)", "V", "osciloscopio", rango=(1.4, 3.45))
    medir("clk_alto", "Senales SPI", "nivel ALTO de CLK (pin 30)", "V", "osciloscopio", rango=(2.1, 3.45))
    medir("clk_bajo", "Senales SPI", "nivel BAJO de CLK (pin 30)", "V", "osciloscopio", rango=(None, 1.1))
    medir("sdi_alto", "Senales SPI", "nivel ALTO de SDI (pin 29)", "V", "osciloscopio", rango=(2.1, 3.45))
    medir("sdi_bajo", "Senales SPI", "nivel BAJO de SDI (pin 29)", "V", "osciloscopio", rango=(None, 1.1))
    medir("clk_f", "Senales SPI", "frecuencia de CLK", "MHz", "osciloscopio",
          como="el servidor pide 1 MHz; el controlador puede entregar menos. El maximo del ADRF es 20 MHz")
else:
    print("(2b se omite en modo ESP32: las senales SPI se miran en el Serial del emulador)")

=== 2b. niveles logicos de las senales SPI (osciloscopio) ===
  Sondas en CS (pin 27), CLK (pin 30) y SDI (pin 29) del U9, con masa corta. Disparo: flanco de bajada de CS.
  [MEDIR] Senales SPI
          nivel BAJO de CS durante una transaccion (pin 27) [V] con osciloscopio
          como: es el nivel que fija R10 (330 ohm) junto con la capacidad de corriente del pin del Red Pitaya
          esperado: como maximo 0.6 V
  [OK] Senales SPI: medido = 0.2 V; esperado: como maximo 0.6 V
  [MEDIR] Senales SPI
          nivel ALTO de CS entre transacciones (pin 27) [V] con osciloscopio
          esperado: entre 1.4 y 3.45 V
  [OK] Senales SPI: medido = 3.31 V; esperado: entre 1.4 y 3.45 V
  [MEDIR] Senales SPI
          nivel ALTO de CLK (pin 30) [V] con osciloscopio
          esperado: entre 2.1 y 3.45 V
  [OK] Senales SPI: medido = 3.32 V; esperado: entre 2.1 y 3.45 V
  [MEDIR] Senales SPI
          nivel BAJO de CLK (pin 30) [V] con osciloscopio
          esperado: como maximo 1.1 V
  [OK]

## 2c. Enganche del PLL con el tester (solo `ADRF_REAL`)

Si el PLL engancha con algún LO, la referencia, el lazo y el chip funcionan; si no engancha con ninguno, el problema está en la referencia, el lazo o la alimentación. Se prueban LO enteros (`FRAC = 0`, sin modulador sigma-delta) y al final el LO de operación. Por cada LO se miden:

- `LDET`: en la pata de `R2` que va al pin 40 del `U9` (no el lado del LED). Enganchado = nivel alto.
- `VTUNE`: en el pad de `C30`. Enganchado = estable y dentro de ~0,9 a 2,5 V (eje de la figura 57 de la hoja de datos).

También se leen de vuelta `INT`, `FRAC` y `RFDIV`, y `CR31` y `CR32` (solo lectura, sin documentar: solo se anotan).

In [ ]:
# ============================================================
# 2c. Enganche del PLL con el tester: LDET, VTUNE y registros de frecuencia
# ============================================================
if MODO == 'ADRF_REAL' and INTERACTIVO:
    print("=== 2c. enganche del PLL: LDET y VTUNE con el tester ===")
    print("  LDET: pata de R2 que va al pin 40 del U9 (no el lado del LED). Enganchado = nivel alto.")
    print("  VTUNE: pad de C30. Enganchado = estable, dentro de ~0,9 a 2,5 V (eje de la figura 57).")

    def lo_entero(lo_aprox_hz):            # LO cercano con FRAC = 0 para la PFD actual
        rf = plan_lo(lo_aprox_hz)['rfdiv']
        return PFD_HZ * round(lo_aprox_hz * 2 ** (rf + 1) / PFD_HZ) // 2 ** (rf + 1)

    PUNTOS_2C_HZ = [lo_entero(151_900_000), lo_entero(337_500_000), lo_entero(810_000_000), LO_DEFECTO_HZ]   # el ultimo deja el LO de operacion
    for lo in PUNTOS_2C_HZ:
        p = plan_lo(lo)
        print(f"\n--- LO = {lo/1e6:g} MHz  (RFDIV={p['rfdiv']}, N={p['n_int']}+{p['n_frac']}/2^25, VCO={p['vco_mhz']} MHz) ---")
        rp(f"ADRF: set_lo({lo} Hz) -> 0")
        ok = rpio.adrf_set_lo(sock, lo)
        chk(f"2c set_lo({lo/1e6:g} MHz): ack True", ok, f"ack={ok}")
        v = rpio.adrf_get_verify(sock)
        if v['retries'] or v['failures']:
            print(f"  Enlace SPI: {v['writes']} escrituras verificadas, {v['retries']} reintentos, {v['failures']} fallas")
        time.sleep(0.5)                    # mucho mas que los ~260 us de adquisicion
        for addr, esperado in regs_lo(p):  # INT, FRAC y RFDIV leidos de vuelta por SDO
            leido = rpio.adrf_read_reg(sock, addr)
            chk(f"2c CR{addr} con LO = {lo/1e6:g} MHz", leido == esperado, f"leido 0x{leido:02X}, esperado 0x{esperado:02X}")
        print(f"  CR31 = 0x{rpio.adrf_read_reg(sock, 31):02X}   CR32 = 0x{rpio.adrf_read_reg(sock, 32):02X}   (solo lectura, sin documentar)")
        medir(f"2c_ldet_{lo}", f"LO = {lo/1e6:g} MHz", "tension en LDET (pata de R2 hacia el pin 40)", "V", "tester en continua",
              como="init ya hecho; contra GND. Enganchado = nivel alto", rango=(2.8, None), orientativo=True)
        medir(f"2c_vtune_{lo}", f"LO = {lo/1e6:g} MHz", "tension en VTUNE (pad de C30)", "V", "tester en continua",
              como="contra GND. Enganchado = estable y dentro de ~0,9 a 2,5 V; sin enganchar suele quedar pegado a un extremo",
              rango=(0.9, 2.5), orientativo=True)

## 2d. Volcado de todos los registros (solo `ADRF_REAL`)

Lee `CR0` a `CR33` y los muestra en hexadecimal y binario, junto a lo que escribe el init. Con `REINICIAR = True` hace antes un init y compara byte a byte; con `False` vuelca el estado tal como está, sin comparar (por ejemplo, después de cambiar el LO o `LOMON`, que difieren del init a propósito). Debajo decodifica lo leído (PLL, PFD, `N`, el LO que resulta de los registros, bomba de carga, autocalibración, detector de lock, `LOMON`, demodulador y VGA) y comprueba que ese LO coincide con el pedido. `CR31` y `CR32` son de solo lectura y no están documentados: solo se anotan.

In [ ]:
# ============================================================
# 2d. Volcado de todos los registros (CR0 a CR33), para depurar
# ============================================================
if MODO == 'ADRF_REAL':
    REINICIAR = True       # True: init previo y comparacion con lo que escribe el init | False: vuelca el estado tal como esta
    print("=== 2d. volcado de registros CR0 a CR33 ===")
    escrito = {}
    if REINICIAR:
        rp(f"ADRF: VGAIN externo (J2): aplicar {vgain_mv(0)} mV",
           f"ADRF: VGAIN externo (J2): aplicar {vgain_mv(GAIN_DEFECTO_MDB)} mV",
           MSG_INIT)
        chk("2d init previo al volcado: ack True", iniciar_adrf())
        escrito = dict(regs_init(p0)); escrito[30] = (VGA_POL << 2) | 1     # CR30 termina asi (paso 33 de la secuencia)
    r = {a: rpio.adrf_read_reg(sock, a) for a in range(rpio.ADRF_REG_MAX + 1)}
    difieren = []
    for a in range(rpio.ADRF_REG_MAX + 1):
        e = escrito.get(a)
        if e is None:
            print(f"  CR{a:<2d}  leido 0x{r[a]:02X} ({r[a]:08b})   " + ("solo lectura" if a >= 31 else ""))
        else:
            difieren += [a] * (r[a] != e)
            print(f"  CR{a:<2d}  leido 0x{r[a]:02X} ({r[a]:08b})   escrito 0x{e:02X}   {'ok' if r[a] == e else '<-- DIFIERE'}")

    # Lo que significan los registros leidos (hoja de datos, tablas 9 a 27)
    doblador, div2, rdiv_en = (r[10] >> 5) & 1, (r[10] >> 6) & 1, (r[5] >> 4) & 1
    rdiv = ((r[10] & 0x1F) or 32) if rdiv_en else 1
    pfd_r = REFIN_HZ * (1 + doblador) / (rdiv * (1 + div2))
    n_int = ((r[7] & 0x0F) << 8) | r[6]
    n_frac = ((r[3] & 1) << 24) | (r[2] << 16) | (r[1] << 8) | r[0]
    rfdiv = r[28] & 7
    lo_r = pfd_r * (n_int + n_frac / 2 ** 25) / 2 ** (rfdiv + 1)
    mA = [0.31, 0.63, 0.94, 1.25, 1.57, 1.88, 2.19, 2.50, 2.81, 3.13, 3.44, 3.75, 4.06, 4.38, 4.69, 5.00]
    cuenta = 4096 if (r[14] >> 7) & 1 else (2048 if (r[23] >> 3) & 1 else 3072)
    print("\n  Decodificado:")
    print(f"    PLL {'APAGADO' if (r[12] >> 2) & 1 else 'encendido'} | REFIN {REFIN_HZ/1e6:g} MHz, doblador {doblador}, div2 {div2}, R {rdiv} -> PFD {pfd_r/1e6:g} MHz")
    print(f"    N = {n_int} + {n_frac}/2^25, RFDIV = {rfdiv} -> LO = {lo_r/1e6:.6f} MHz")
    print(f"    bomba de carga {mA[r[9] >> 4]} mA | autocalibracion {'deshabilitada' if r[24] & 1 else 'habilitada'}, {r[25] * 24 / pfd_r * 1e6:.0f} us")
    print(f"    detector de lock {'habilitado' if (r[23] >> 4) & 1 else 'DESHABILITADO'}, {cuenta} pulsos, precision {'fina' if (r[23] >> 2) & 1 else 'gruesa'}")
    print(f"    LOMON {'encendido' if (r[27] >> 2) & 1 else 'apagado'} ({[-24, -18, -12, -6][r[27] & 3]} dBm) | demodulador {'encendido' if r[29] & 1 else 'APAGADO'}, "
          f"VOCM {'interno' if (r[29] >> 6) & 1 else 'externo'}, {'wideband' if (r[29] >> 3) & 1 else 'narrow-band ' + str([50, 43, 37, 30][(r[29] >> 4) & 3]) + ' MHz'}")
    print(f"    VGA {'encendido' if r[30] & 1 else 'APAGADO'}, pendiente {'negativa' if (r[30] >> 2) & 1 else 'positiva'} | revision (CR33) 0x{r[33]:02X}")
    if REINICIAR:
        chk("2d registros leidos = los que escribe el init (byte completo, CR0 a CR30)", not difieren, f"difieren: {['CR%d' % a for a in difieren]}" if difieren else "todos coinciden")
        chk("2d el LO que resulta de los registros coincide con el pedido", abs(lo_r - p0['lo_actual']) < 1, f"{lo_r:.3f} Hz leido, {p0['lo_actual']} Hz pedido")

## 3. LO monitor encendido

`LOMON` es la forma de **ver el LO** desde afuera: una réplica a la frecuencia del LO. Queda encendido durante el barrido del paso 4 y hasta el paso 6, que lo recorre y lo apaga.

In [27]:
print("=== 3. LOMON encendido (-12 dBm) ===")
rp("ADRF: update (mask=0x08) -> 0")
esp(wr(27, 0x06))
ok = rpio.adrf_update(sock, rpio.ADRF_UPD_LOMON, lomon_en=1, lomon_pwr=rpio.ADRF_LOMON_M12DBM)
chk("update LOMON: ack True", ok, f"ack={ok}")
if MODO == 'ESP32':
    manual("Serial: WRITE CR27 = 0x06 con 'LOMON on, -12 dBm'")
else:
    medir("lomon_f0", "LOMON encendido", f"frecuencia del LO en LOMON, SMA J4 ({LO_DEFECTO_HZ/1e6:g} MHz)", "MHz", "analizador de espectro en J4",
          como="la exactitud en frecuencia es la del oscilador U2 mas la del instrumento",
          esperado=plan_lo(LO_DEFECTO_HZ)['lo_actual'] / 1e6, tol_ppm=TOL_FREQ_PPM, orientativo=True)
    medir("lomon_p0", "LOMON encendido", "nivel en LOMON (J4), con -12 dBm programado", "dBm", "analizador de espectro en J4",
          como=("solo se anota: segun la netlist, J4 toma LOMON a traves de 56 pF (C43) y la corriente se reparte con la resistencia de polarizacion (R16), "
                "asi que el nivel en J4 puede quedar varios dB por debajo del nominal de la hoja de datos; restar ademas las perdidas del cable. "
                "Lo que se verifica en la seccion 6 son los pasos de ~6 dB entre niveles"))

=== 3. LOMON encendido (-12 dBm) ===
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: update (mask=0x08) -> 0
  [OK] update LOMON: ack True: ack=True
  [MEDIR] LOMON encendido
          frecuencia del LO en LOMON, SMA J4 (151.275 MHz) [MHz] con analizador de espectro en J4
          como: la exactitud en frecuencia es la del oscilador U2 mas la del instrumento
          esperado: 151.275 MHz +/- 0.0151275 (100 ppm)  (criterio orientativo propuesto, no es especificacion)
    -> OMITIDA
  [MEDIR] LOMON encendido
          nivel en LOMON (J4), con -12 dBm programado [dBm] con analizador de espectro en J4
          como: solo se anota: segun la netlist, J4 toma LOMON a traves de 56 pF (C43) y la corriente se reparte con la resistencia de polarizacion (R16), asi que el nivel en J4 puede quedar varios dB por debajo del nominal de la hoja de datos; restar ademas las perdidas del cable. Lo que se verifica en la seccion 6 son los pasos de ~6 dB entre niveles
          esper

## 4. Barrido de LO

Un punto de cada banda de `RFDIV` (Tabla 6), los **bordes** de banda (500, 250 y 125 MHz), los extremos del rango (100 y 1000 MHz) y vuelta al valor por defecto. El `INT`/`FRAC`/`RFDIV` esperado se calcula acá con las ecuaciones de la hoja de datos y se compara con el del driver; en `ADRF_REAL` además se mide la frecuencia real en `LOMON`.

In [33]:
PUNTOS_LO_HZ = [157_025_000]

print("=== 4. barrido de LO ===")
for lo in PUNTOS_LO_HZ:
    p = plan_lo(lo)
    print()
    print(f"--- LO = {lo/1e6:g} MHz  (RFDIV={p['rfdiv']}, N={p['n_int']}+{p['n_frac']}/2^25, VCO={p['vco_mhz']} MHz) ---")
    rp(f"ADRF: set_lo({lo} Hz) -> 0")
    esp(*lineas_lo_esp32(p), laxas=["LDET -> 1"])
    ok = rpio.adrf_set_lo(sock, lo)
    est = rpio.adrf_get_state(sock)
    chk(f"set_lo({lo/1e6:g} MHz): ack True", ok, f"ack={ok}")
    comparar_estado(est, estado_esperado(lo_hz=lo), ['lo_hz', 'rfdiv', 'n_int', 'n_frac', 'lo_actual_hz'])
    if MODO == 'ESP32':
        manual(f"LED D3 encendido con LO = {lo/1e6:g} MHz")
    else:
        manual(f"LED D3 encendido con LO = {lo/1e6:g} MHz (baja un instante durante la adquisicion, ~260 us)")
        medir(f"lo_{lo}", f"LO = {lo/1e6:g} MHz", "frecuencia en LOMON (J4)", "MHz", "analizador de espectro o contador en J4",
              como="LOMON (J4) encendido a -12 dBm (paso 3)",
              esperado=p['lo_actual'] / 1e6, tol_ppm=TOL_FREQ_PPM, orientativo=True)

=== 4. barrido de LO ===

--- LO = 157.025 MHz  (RFDIV=2, N=93+1739859/2^25, VCO=2512 MHz) ---
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: set_lo(157025000 Hz) -> 0
  [OK] set_lo(157.025 MHz): ack True: ack=True
  [OK] estado.lo_hz: esperado=157025000  real=157025000
  [OK] estado.rfdiv: esperado=2  real=2
  [OK] estado.n_int: esperado=93  real=93
  [OK] estado.n_frac: esperado=1739859  real=1739859
  [OK] estado.lo_actual_hz: esperado=157025000  real=157025000
  [MANUAL] LED D3 encendido con LO = 157.025 MHz (baja un instante durante la adquisicion, ~260 us)


  [MEDIR] LO = 157.025 MHz
          frecuencia en LOMON (J4) [MHz] con analizador de espectro o contador en J4
          como: LOMON (J4) encendido a -12 dBm (paso 3)
          esperado: 157.025 MHz +/- 0.0157025 (100 ppm)  (criterio orientativo propuesto, no es especificacion)
    -> OMITIDA


## 5. Barrido de ganancia

`adrf_set_gain` **no genera tráfico SPI ni cambia la ganancia real**: el driver solo calcula la tensión de `VGAIN` (25 mV/dB) y la informa en la consola del servidor. El notebook la aplica por el DAC (`VGAIN_POR_DAC`); en `ADRF_REAL` verificás con el tester que en `J2` esté la tensión pedida (valida la calibración del DAC) y medís el tono de banda base: cada paso de 250 mV debe subir el nivel unos 10 dB. Los puntos cuya tensión el DAC no alcanza se omiten. La portadora de RF de prueba tiene que caer dentro del filtro de entrada de la placa (156 a 162 MHz): por eso en `ADRF_REAL` este barrido pone el LO en 151,975 MHz (RF = 161,975 MHz) y no en 330 MHz. La hoja de datos garantiza la conformidad de ganancia (0,5 dB) solo con `VGAIN` entre 200 mV y 1,3 V, así que 0 mV y 1500 mV se anotan sin criterio de aprobación.

In [46]:
PUNTOS_GANANCIA_MDB = [0, 10_000, 20_000, 30_000, 40_000, 50_000, 60_000]
en_rango_conf = lambda mv: 200 <= mv <= 1300       # rango de conformidad de ganancia (hoja de datos)

print(f"=== 5. barrido de ganancia (VGAIN = 25 mV/dB, pendiente {'negativa' if VGA_POL else 'positiva'}) ===")
if LAB:
    # La entrada de RF de la placa tiene un filtro de 156 a 162 MHz (FB1): una portadora fuera de esa banda llegaria muy atenuada.
    # Por eso el LO de este barrido no es el de 330 MHz sino el que deja la RF de prueba dentro de la banda.
    LO_GAN_HZ = F_RF_PRUEBA_HZ - int(F_BB_MHZ * 1_000_000)
    print(f"  Entrada de RF (J1): portadora de {F_RF_PRUEBA_HZ/1e6:.6f} MHz, {RF_ENTRADA_DBM} dBm fijos (sugerencia).")
    print(f"  Se pone el LO en {LO_GAN_HZ/1e6:.6f} MHz para que la portadora caiga en un tono de {F_BB_MHZ} MHz, que se mide en J5 (I) o J6 (Q).")
    rp(f"ADRF: set_lo({LO_GAN_HZ} Hz) -> 0")
    ok = rpio.adrf_set_lo(sock, LO_GAN_HZ)
    chk("set_lo (LO del barrido de ganancia): ack True", ok, f"ack={ok}")
    comparar_estado(rpio.adrf_get_state(sock), estado_esperado(lo_hz=LO_GAN_HZ), ['lo_hz', 'rfdiv', 'n_int', 'n_frac', 'lo_actual_hz'])
    manual(f"LED D3 encendido con LO = {LO_GAN_HZ/1e6:.6f} MHz")
prev_mv, prev_clave = None, None
for g in PUNTOS_GANANCIA_MDB:
    mv = vgain_mv(g)
    print()
    print(f"--- ganancia = {g/1000:g} dB -> VGAIN {mv} mV ---")
    rp(f"ADRF: VGAIN externo (J2): aplicar {mv} mV",
       f"ADRF: set_gain({g} mdB) -> 0")
    esp(laxas=["sin lineas nuevas: el cambio de ganancia no genera trafico SPI",
               f"(opcional, comando 'g' con {mv} mV en GPIO34) ganancia ~ {g/1000:g} dB"])
    ok = rpio.adrf_set_gain(sock, g)
    est = rpio.adrf_get_state(sock)
    chk(f"set_gain({g/1000:g} dB): ack True", ok, f"ack={ok}")
    comparar_estado(est, estado_esperado(lo_hz=LO_DEFECTO_HZ, gain_mdb=g), ['gain_mdb', 'vgain_mv'])
    efectivo_mv = mv
    if VGAIN_POR_DAC:
        v = rpio.adrf_get_vgain(sock)
        efectivo_mv = v['efectivo_mv']
        if efectivo_mv != mv:
            print(f"  AVISO: el DAC aplica {efectivo_mv} mV en lugar de {mv} mV (satura): la ganancia real difiere en {abs(mv - efectivo_mv) / 25:.1f} dB")
    if prev_mv is not None:
        sube = efectivo_mv < prev_mv if VGA_POL else efectivo_mv > prev_mv    # con pendiente negativa la tension BAJA al subir la ganancia
        chk("VGAIN " + ("decrece" if VGA_POL else "crece") + " con la ganancia (pendiente " + ("negativa" if VGA_POL else "positiva") + ")",
            sube, f"{prev_mv} -> {efectivo_mv} mV")
    if MODO == 'ESP32':
        manual("Serial sin lineas nuevas (no hay trafico SPI en un cambio de ganancia)")
    else:
        medir(f"vj2_{mv}", f"Ganancia {g/1000:g} dB", "tension realmente aplicada en J2", "mV", "tester",
              como=(f"el DAC aplica {efectivo_mv} mV: verificar con el tester; esto valida la calibracion del DAC" if VGAIN_POR_DAC
                    else f"ajustar la fuente a {mv} mV y verificar; esto valida el montaje"), esperado=efectivo_mv, tol=10, orientativo=True)
        con_criterio = prev_mv is not None and en_rango_conf(mv) and en_rango_conf(prev_mv)
        medir(f"bb_{mv}", f"Ganancia {g/1000:g} dB", f"nivel del tono de {F_BB_MHZ} MHz en banda base", "dBm", "analizador de espectro en J5 o J6",
              como="mismo nivel de entrada de RF en todo el barrido; si el tono se pierde en el ruido o satura, cambiar RF_ENTRADA_DBM",
              rel=(prev_clave, "dif", 10.0, 1.0) if con_criterio else None, orientativo=True)
    prev_mv, prev_clave = efectivo_mv, f"bb_{mv}"

rpio.adrf_set_gain(sock, GAIN_DEFECTO_MDB)     # vuelve al valor por defecto

=== 5. barrido de ganancia (VGAIN = 25 mV/dB, pendiente positiva) ===
  Entrada de RF: portadora a 340 MHz (LO + 10 MHz), -60 dBm fijos (sugerencia).
  Se mide el tono de 10 MHz en una pata de I o Q de la banda base (acoplada en AC).

--- ganancia = 0 dB -> aplicar 0 mV en J2 ---
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: VGAIN externo (J2): aplicar 0 mV
    RP    > ADRF: set_gain(0 mdB) -> 0
  [OK] set_gain(0 dB): ack True: ack=True
  [OK] estado.gain_mdb: esperado=0  real=0
  [OK] estado.vgain_mv: esperado=0  real=0
  [MEDIR] Ganancia 0 dB
          tension realmente aplicada en J2 [mV] con tester
          como: ajustar la fuente a 0 mV y verificar; esto valida el montaje
          esperado: 0 mV +/- 10  (criterio orientativo propuesto, no es especificacion)
  [OK] Ganancia 0 dB: medido = 3 mV; esperado: 0 mV +/- 10
  [MEDIR] Ganancia 0 dB
          nivel del tono de 10 MHz en banda base [dBm] con analizador de espectro
          como: mismo nivel de entra

True

## 6. Niveles del LO monitor

Recorre los cuatro niveles de salida de `LOMON` y comprueba que cada uno queda unos 6 dB por encima del anterior; termina apagándolo. En uso normal `LOMON` queda apagado: esta prueba solo verifica el camino de escritura de `CR27`. El corte del filtro de banda base y la corriente de la bomba de carga quedan en valores fijos (ver el archivo de configuración) y no se prueban acá.

In [33]:
print("=== 6. niveles del LO monitor (CR27) ===")
niveles = [(1, 0, "-24 dBm", -24), (1, 1, "-18 dBm", -18), (1, 2, "-12 dBm", -12), (1, 3, "-6 dBm", -6), (0, 0, "apagado", None)]
prev_clave = None
for en, pwr, nombre, dbm in niveles:
    cr27 = (en << 2) | pwr
    print()
    print(f"--- LOMON {nombre} ---")
    rp("ADRF: update (mask=0x08) -> 0")
    esp(wr(27, cr27))
    ok = rpio.adrf_update(sock, rpio.ADRF_UPD_LOMON, lomon_en=en, lomon_pwr=pwr)
    chk(f"update LOMON ({nombre}): ack True", ok, f"ack={ok}")
    if MODO == 'ESP32':
        manual(f"Serial: WRITE CR27 = 0x{cr27:02X}")
    elif dbm is None:
        manual("LOMON sin senal (apagado)")
    else:
        clave = f"lomon_n_{dbm}"
        medir(clave, f"LOMON {nombre}", "nivel en LOMON (J4)", "dBm", "analizador de espectro en J4",
              como=("el primer nivel solo se anota; los siguientes se comparan con el anterior: los cuatro niveles estan separados unos 6 dB "
                    "(hoja de datos: aprox. -6/-12/-18/-24 dBm). El nivel absoluto en J4 depende del acoplamiento de la placa"),
              rel=None if prev_clave is None else (prev_clave, "dif", 6.0, 1.5), orientativo=True)
        prev_clave = clave
print()
print("(El estado del driver no tiene campo de LOMON: no hay verificacion automatica de su estado.)")

=== 6d. niveles del LO monitor (CR27) ===

--- LOMON -24 dBm ---
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: update (mask=0x08) -> 0
  ESPERADO en el Serial del ESP32:
    ESP32 > WRITE CR27 (0x1B) = 0x04
  [OK] update LOMON (-24 dBm): ack True: ack=True
  [MANUAL] Serial: WRITE CR27 = 0x04

--- LOMON -18 dBm ---
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: update (mask=0x08) -> 0
  ESPERADO en el Serial del ESP32:
    ESP32 > WRITE CR27 (0x1B) = 0x05
  [OK] update LOMON (-18 dBm): ack True: ack=True
  [MANUAL] Serial: WRITE CR27 = 0x05

--- LOMON -12 dBm ---
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: update (mask=0x08) -> 0
  ESPERADO en el Serial del ESP32:
    ESP32 > WRITE CR27 (0x1B) = 0x06
  [OK] update LOMON (-12 dBm): ack True: ack=True
  [MANUAL] Serial: WRITE CR27 = 0x06

--- LOMON -6 dBm ---
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: update (mask=0x08) -> 0
  ESPERADO en 

## 7. Reinicialización

El chip ya está en `SPI` (solo se vuelve a `I2C` cortando la alimentación), así que los `3` pulsos de `CS` son redundantes y el chip los ignora. Se valida que **los registros** vuelven a los valores por defecto.

In [34]:
print("=== 7. re-init ===")
rp(f"ADRF: VGAIN externo (J2): aplicar {vgain_mv(0)} mV",
   f"ADRF: VGAIN externo (J2): aplicar {vgain_mv(GAIN_DEFECTO_MDB)} mV",
   MSG_INIT)
esp(*lineas_init_esp32(p0, reinit=True), laxas=["LDET -> 1"])
ok = iniciar_adrf()
chk("re-init: ack True", ok, f"ack={ok}")
comparar_estado(rpio.adrf_get_state(sock), estado_esperado())
if MODO == 'ESP32':
    manual("Serial: 3 'CS pulso extra', secuencia completa y 'INIT: secuencia COMPLETA y conforme'")
else:
    manual("LED D3 encendido tras el re-init (baja un instante)")
    manual("LOMON sin senal (el init lo deja apagado: CR27 = 0x00)")
    medir_corriente("i_reinit", "Despues del re-init")

=== 7. re-init ===
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: VGAIN externo (J2): aplicar 0 mV
    RP    > ADRF: VGAIN externo (J2): aplicar 500 mV
    RP    > ADRF: init -> 0
  ESPERADO en el Serial del ESP32:
    ESP32 > CS pulso extra tras el lock (1 byte 0x00): el chip lo ignora
    ESP32 > CS pulso extra tras el lock (1 byte 0x00): el chip lo ignora
    ESP32 > CS pulso extra tras el lock (1 byte 0x00): el chip lo ignora
    ESP32 > WRITE CR30 (0x1E) = 0x00
    ESP32 > INIT: comienzo de secuencia detectado (CR30 con VGA apagado)
    ESP32 > WRITE CR29 (0x1D) = 0x41
    ESP32 > WRITE CR28 (0x1C) = 0x09
    ESP32 > WRITE CR27 (0x1B) = 0x00
    ESP32 > WRITE CR26 (0x1A) = 0x00
    ESP32 > WRITE CR25 (0x19) = 0x70
    ESP32 > WRITE CR24 (0x18) = 0x38
    ESP32 > WRITE CR23 (0x17) = 0x70
    ESP32 > WRITE CR22 (0x16) = 0x00
    ESP32 > WRITE CR21 (0x15) = 0x00
    ESP32 > WRITE CR20 (0x14) = 0x00
    ESP32 > WRITE CR19 (0x13) = 0x00
    ESP32 > WRITE CR18 (0x

## 7b. Configuración para recibir AIS (LO por debajo de la señal, FI de 10,7 MHz)

Baja el canal A de AIS (161,975 MHz) a una FI de 10,7 MHz poniendo el LO **por debajo** de la señal: `LO = F_AIS − 10,7 MHz` (151,275 MHz). El init ya deja el ADRF en este punto de operación, porque sale del mismo `.conf`: esta sección lo vuelve a aplicar, lo mide en el banco y permite probar una corrección de la referencia (`CORR_PPM`) sin tocar el `.conf`. El ADRF6850 es un demodulador en cuadratura: el tono de FI sale en las salidas I y Q de banda base.

Notas para el montaje:

- **Imagen**: con LO bajo, lo que entre en `F_AIS − 21,4 MHz` (140,575 MHz) también cae en 10,7 MHz. La placa tiene en la entrada de RF (`J1`) un filtro de 156 a 162 MHz (`FB1`, según la netlist) que debería rechazarla: medí su respuesta en 140,575 MHz y en 161,975 y 162,025 MHz (este último queda justo en el borde de su banda). Si se usa **una sola salida** (`J5`, señal real) el chip no rechaza la imagen; con **`J5` y `J6`** a la vez (I y Q, señal compleja) el rechazo depende del balance I/Q del ADRF.
- **Exactitud del LO**: cada ppm de la referencia `U2` desplaza el tono unos 151 Hz. Con ±25 a ±50 ppm son ±3,8 a ±7,6 kHz, más de lo que se probó con buen resultado (hasta ±1 kHz). En `ADRF_REAL` se mide en `LOMON` y se corrige con `CORR_PPM` (la propia celda calcula el valor).
- **Salidas de banda base**: `J5` (I) y `J6` (Q) ya son de simple terminación: baluns `T2`/`T3` y, en cada salida, un pasabajos LC con carga de 50 Ω (`R19`/`R20`), así que no hace falta un balun externo. Según la netlist, sus valores coinciden con un Butterworth de 5.º orden de unos 15 MHz para 50 Ω (cálculo ideal, sin el balun): deja pasar los 10,7 MHz casi sin pérdida (~0,1 dB) y atenúa unos 60 dB en 62,5 MHz (Nyquist del ADC) y unos 88 dB en 114,3 MHz (el alias de 10,7 MHz). Medí su respuesta real y la amplitud en el propio ADC.
- **Ganancia**: `VGAIN` lo aplica solo el DAC por registro (`OUT2` → `J2`, servidor con `-d`) con la tensión que calcula el driver; con `VGAIN_POR_DAC = False` hay que aplicarla a mano.

In [35]:
# ============================================================
# 7b. Configuracion para recibir AIS: parametros
# ============================================================
# Los parametros salen de la configuracion. Para probar una correccion de la referencia sin tocarla, cambiar CORR_PPM
# abajo; el valor definitivo va en corr_ppm del .conf de la Red Pitaya (o en CONF_CLIENTE), y se reinicia el servidor.
F_FI_HZ   = F_FI_CONF if F_FI_CONF is not None else 10_700_000
F_AIS_HZ  = F_RF_CONF if F_RF_CONF is not None else F_RF_PRUEBA_HZ
CORR_PPM  = CORR_PPM_CONF                 # error de la referencia U2 medido en LOMON (la celda siguiente lo calcula)
GANANCIA_AIS_MDB = GAIN_DEFECTO_MDB       # la del .conf
BB_FC_AIS = BB_FC_DEF                     # el filtro de banda base del .conf

LO_AIS_NOM_HZ = F_AIS_HZ - F_FI_HZ                          # LO bajo: la senal queda en +FI
LO_AIS_HZ     = (LO_DEFECTO_HZ if CORR_PPM == CORR_PPM_CONF           # sin tocar CORR_PPM: exactamente el LO del .conf
                 else int(LO_AIS_NOM_HZ / (1 + CORR_PPM * 1e-6) + 0.5))   # lo que se le pide al ADRF
assert 100_000_000 <= LO_AIS_HZ <= 1_000_000_000, "LO fuera del rango del ADRF6850 (100 MHz a 1000 MHz)"

p_ais = plan_lo(LO_AIS_HZ)
paso_lo_hz = PFD_HZ / (1 << 25) / (1 << (p_ais['rfdiv'] + 1))
print(f"Senal: RF = {F_AIS_HZ/1e6:.6f} MHz -> FI = {F_FI_HZ/1e6:g} MHz  (de la configuracion: {CONF_ID}" + ("" if CORR_PPM == CORR_PPM_CONF else f"; CORR_PPM de prueba = {CORR_PPM:+g}") + ")")
print(f"  LO pedido = {LO_AIS_HZ/1e6:.6f} MHz  (RFDIV={p_ais['rfdiv']}, N={p_ais['n_int']}+{p_ais['n_frac']}/2^25, VCO={p_ais['vco_mhz']} MHz, paso de LO = {paso_lo_hz:.3f} Hz)")
print(f"  Frecuencia imagen = {(LO_AIS_HZ - F_FI_HZ)/1e6:.3f} MHz  |  error del tono por cada ppm de la referencia = {LO_AIS_HZ*1e-6:.0f} Hz")
print(f"  Ganancia inicial = {GANANCIA_AIS_MDB/1000:g} dB -> VGAIN = {vgain_mv(GANANCIA_AIS_MDB)} mV  |  banda base: {'wideband' if BB_WIDE_DEF else 'narrow-band'}, corte {[50, 43, 37, 30][BB_FC_AIS]} MHz")



Canal A: RF = 161.975 MHz -> FI = 10.7 MHz
  LO pedido = 151.275000 MHz  (RFDIV=2, N=44+27589200/2^25, VCO=2420 MHz, paso de LO = 0.101 Hz)
  Frecuencia imagen = 140.575 MHz  |  error del tono por cada ppm de la referencia = 151 Hz
  Ganancia inicial = 40 dB -> VGAIN = 1000 mV  |  banda base: narrow-band, corte 30 MHz


In [36]:
# ============================================================
# 7c. Configuracion para recibir AIS: ejecucion
# ============================================================
print("=== 7b. configuracion para AIS ===")

print()
print(f"--- 1) LO = {LO_AIS_HZ/1e6:.6f} MHz ---")
rp(f"ADRF: set_lo({LO_AIS_HZ} Hz) -> 0")
esp(*lineas_lo_esp32(p_ais), laxas=["LDET -> 1"])
ok = rpio.adrf_set_lo(sock, LO_AIS_HZ)
est = rpio.adrf_get_state(sock)
chk("set_lo(AIS): ack True", ok, f"ack={ok}")
comparar_estado(est, estado_esperado(lo_hz=LO_AIS_HZ), ['lo_hz', 'rfdiv', 'n_int', 'n_frac', 'lo_actual_hz'])
manual(f"LED D3 encendido con LO = {LO_AIS_HZ/1e6:.6f} MHz")

print()
print(f"--- 2) banda base: narrow-band, corte {[50, 43, 37, 30][BB_FC_AIS]} MHz ---")
rp("ADRF: update (mask=0x04) -> 0")
esp(wr(29, (VOCM_INT << 6) | (BB_FC_AIS << 4) | (int(BB_WIDE_DEF) << 3) | 1))
ok = rpio.adrf_update(sock, rpio.ADRF_UPD_BB, bb_wideband=int(BB_WIDE_DEF), bb_fc=BB_FC_AIS)
est = rpio.adrf_get_state(sock)
chk("update BB (AIS): ack True", ok, f"ack={ok}")
comparar_estado(est, estado_esperado(lo_hz=LO_AIS_HZ, bb_fc=BB_FC_AIS), ['bb_wideband', 'bb_fc'])

print()
mv_ais = vgain_mv(GANANCIA_AIS_MDB)
print(f"--- 3) ganancia {GANANCIA_AIS_MDB/1000:g} dB -> VGAIN {mv_ais} mV ---")
rp(f"ADRF: VGAIN externo (J2): aplicar {mv_ais} mV", f"ADRF: set_gain({GANANCIA_AIS_MDB} mdB) -> 0")
esp(laxas=["sin lineas nuevas: la ganancia no genera trafico SPI"])
ok = rpio.adrf_set_gain(sock, GANANCIA_AIS_MDB)
est = rpio.adrf_get_state(sock)
chk("set_gain(AIS): ack True", ok, f"ack={ok}")
comparar_estado(est, estado_esperado(lo_hz=LO_AIS_HZ, gain_mdb=GANANCIA_AIS_MDB, bb_fc=BB_FC_AIS), ['gain_mdb', 'vgain_mv'])

print()
print("--- estado final de la configuracion AIS ---")
comparar_estado(est, estado_esperado(lo_hz=LO_AIS_HZ, gain_mdb=GANANCIA_AIS_MDB, bb_fc=BB_FC_AIS))

if LAB:
    print()
    print("--- mediciones de banco para AIS ---")
    medir("ais_vj2", "Ganancia para AIS", "tension realmente aplicada en J2", "mV", "tester",
          como=(f"el DAC aplica {rpio.adrf_get_vgain(sock)['efectivo_mv']} mV: verificar con el tester" if VGAIN_POR_DAC else f"ajustar la fuente a {mv_ais} mV y verificar"),
          esperado=rpio.adrf_get_vgain(sock)['efectivo_mv'] if VGAIN_POR_DAC else mv_ais, tol=10, orientativo=True)
    # LO: LOMON se enciende solo para medirlo y se apaga despues
    rpio.adrf_update(sock, rpio.ADRF_UPD_LOMON, lomon_en=1, lomon_pwr=rpio.ADRF_LOMON_M12DBM)
    f_lo = medir("ais_lo", "LO para AIS", "frecuencia en LOMON", "MHz", "analizador de espectro o contador",
                 como="LOMON (J4) encendido a -12 dBm solo para esta medicion",
                 esperado=p_ais['lo_actual'] / 1e6, tol_ppm=TOL_FREQ_PPM, orientativo=True)
    rpio.adrf_update(sock, rpio.ADRF_UPD_LOMON, lomon_en=0, lomon_pwr=0)
    if f_lo is not None:
        e_ppm = (f_lo * 1e6 - LO_AIS_HZ) / LO_AIS_HZ * 1e6
        print(f"  Error de la referencia U2: {e_ppm:+.1f} ppm ({f_lo*1e6 - LO_AIS_HZ:+,.0f} Hz a {LO_AIS_HZ/1e6:.3f} MHz).")
        print(f"  -> para corregirlo poner CORR_PPM = {e_ppm:+.1f} (no sumarlo a un valor previo) y volver a correr 7b.")
    print(f"  Aplicar ahora en la entrada de RF (J1) un tono de {F_AIS_HZ/1e6:.6f} MHz (empezar con nivel bajo).")
    medir("ais_fi", "Tono de prueba", "frecuencia del tono en banda base (J5, I)", "MHz", "analizador de espectro o FFT del osciloscopio",
          como=f"debe caer en {F_FI_HZ/1e6:g} MHz; el error es el del LO mas el del generador (usar la misma referencia si se puede)",
          esperado=F_FI_HZ / 1e6, tol=LO_AIS_HZ * TOL_FREQ_PPM * 1e-6 / 1e6, orientativo=True)
    medir("ais_vpp", "Tono de prueba", "amplitud del tono en la salida que va al ADC (J5 o J6)", "Vpp", "osciloscopio",
          como="con el jumper LV del Red Pitaya el ADC admite +/-1 V (2 Vpp): si pasa de ahi, bajar la ganancia o el nivel de RF. Verificar el jumper del equipo",
          rango=(None, 1.8))
    medir("ais_iq_fase", "Cuadratura I/Q", "diferencia de fase entre Q e I (valor absoluto)", "grados", "osciloscopio de dos canales",
          como="con el mismo tono; ingresar el valor absoluto", esperado=90, tol=10, orientativo=True)
    manual("Anotar el SIGNO de la fase: si Q adelanta o atrasa a I con la RF por encima del LO. Define si hay que intercambiar I y Q al formar I + jQ en la FPGA")
    medir("ais_iq_amp", "Balance I/Q", "diferencia de nivel entre I y Q", "dB", "osciloscopio o analizador de espectro",
          como="con el mismo tono", esperado=0, tol=1, orientativo=True)

=== 7b. configuracion para AIS ===

--- 1) LO = 151.275000 MHz ---
  ESPERADO en la consola del servidor (Red Pitaya):
    RP    > ADRF: set_lo(151275000 Hz) -> 0
  ESPERADO en el Serial del ESP32:
    ESP32 > WRITE CR28 (0x1C) = 0x0A
    ESP32 > WRITE CR7  (0x07) = 0x00
    ESP32 > WRITE CR6  (0x06) = 0x2C
    ESP32 > WRITE CR3  (0x03) = 0x01
    ESP32 > WRITE CR2  (0x02) = 0xA4
    ESP32 > WRITE CR1  (0x01) = 0xFA
    ESP32 > WRITE CR0  (0x00) = 0x50
    ESP32 > ACQ   (CR0) PFD=27.000 MHz  N=44+27589200/2^25  RFDIV=2  LO=151.275000 MHz  VCO=2420 MHz  autocal on (99 us)
    ESP32 > LDET: engancha en ~260 us
    ESP32 ~ LDET -> 1
  [OK] set_lo(AIS): ack True: ack=True
  [OK] estado.lo_hz: esperado=151275000  real=151275000
  [OK] estado.rfdiv: esperado=2  real=2
  [OK] estado.n_int: esperado=44  real=44
  [OK] estado.n_frac: esperado=27589200  real=27589200
  [OK] estado.lo_actual_hz: esperado=151275000  real=151275000
  [MANUAL] LED D3 encendido con LO = 151.275000 MHz

--- 2) banda b

## 8. Resumen y registro de la prueba

In [37]:
n_ok = sum(1 for _, ok in RES if ok)
print(f"Verificaciones automaticas: {n_ok}/{len(RES)} OK")
for nombre, ok in RES:
    if not ok:
        print(f"  [FALLO] {nombre}")
print()
pend = [d for d, r in MANUALES if r is None]
mal  = [d for d, r in MANUALES if r is False]
print(f"Verificaciones manuales: {len(MANUALES)} en total, {len(pend)} sin confirmar, {len(mal)} marcadas como NO cumplidas")
for d in mal:
    print(f"  [NO CUMPLIDA] {d}")
if pend and not INTERACTIVO:
    print("  (INTERACTIVO = False: las verificaciones manuales solo se listaron; revisalas contra el Serial del ESP32 / el banco)")
_csv_fila("resumen", "verificaciones automaticas", resultado=f"{n_ok}/{len(RES)} OK")
_csv_fila("resumen", "verificaciones manuales",
          resultado=f"{len(MANUALES) - len(pend) - len(mal)} cumplidas, {len(mal)} no cumplidas, {len(pend)} sin confirmar")

if MEDS:
    print()
    print("Mediciones de banco:")
    print(f"  {'etapa':<48} {'magnitud':<46} {'medido':>10} {'unid.':<5} {'veredicto':<10}")
    for m in MEDS:
        med = "-" if m['medido'] is None else f"{m['medido']:g}"
        print(f"  {m['etapa'][:47]:<48} {m['magnitud'][:45]:<46} {med:>10} {m['unidad']:<5} {m['veredicto']:<10}")
    fuera = [m for m in MEDS if m['veredicto'] == 'FUERA']
    print()
    print(f"  {sum(m['veredicto'] == 'OK' for m in MEDS)} OK, {len(fuera)} FUERA de criterio, "
          f"{sum(m['veredicto'] == 'ANOTADA' for m in MEDS)} anotadas, "
          f"{sum(m['veredicto'] in ('OMITIDA', 'NO MEDIDA', 'SIN MEDIR') for m in MEDS)} sin valor")
    for m in fuera:
        print(f"  [FUERA] {m['etapa']} / {m['magnitud']}: medido {m['medido']:g} {m['unidad']}; {m['criterio']}"
              + (f"  -- comentario: {m['comentario']}" if m['comentario'] else ""))
    _csv_fila("resumen", "mediciones de banco",
              resultado=f"{sum(m['veredicto'] == 'OK' for m in MEDS)} OK, {len(fuera)} FUERA, "
                        f"{sum(m['veredicto'] == 'ANOTADA' for m in MEDS)} anotadas, "
                        f"{sum(m['veredicto'] in ('OMITIDA', 'NO MEDIDA', 'SIN MEDIR') for m in MEDS)} sin valor")

print()
print(f"Resultados guardados en: {os.path.abspath(ARCHIVO_CSV)}")
print("  (CSV para Excel en espanol: separador ';' y coma decimal; se fue escribiendo durante la prueba)")
print()
print("Recordatorio: las verificaciones de las secciones 1 y 2 usan la copia interna del driver; la que prueba el enlace es la 2a (lectura por SDO).")
print("Ademas, el Serial del ESP32 o el LED D3 y las mediciones confirman que el chip recibio los comandos.")
if VGAIN_POR_DAC:
    rpio.adrf_vgain_raw(sock, vgain_seguro_raw())      # deja VGAIN en la ganancia minima
sock.close()


Verificaciones automaticas: 180/180 OK

Verificaciones manuales: 29 en total, 29 sin confirmar, 0 marcadas como NO cumplidas

Recordatorio: las verificaciones automaticas no pueden detectar un enlace SPI muerto;
la evidencia de que el chip recibio los comandos es el Serial del ESP32 o el LED D3 / las mediciones.
